# Local Machine Snowflake Connection

In [430]:
import pandas as pd
import numpy as np
#import pyodbc
import glob
import os
import re
import snowflake.connector
import urllib.parse
from sqlalchemy import create_engine, text
from getpass import getpass
from openpyxl import load_workbook

In [431]:
user_nm = 'jingna.liu'

In [432]:
sfAccount: str =  'merkle-amica' 
sfauthenticator: str = 'externalbrowser'

ctx = snowflake.connector.connect(
    user= f'{user_nm}@merkleinc.com', #enter your user name
    account=sfAccount,
    authenticator=sfauthenticator,
    role = 'AMICA_SNOWFLAKE_PROD_NON_PII_CAMPAIGN' #other options are: 'AMICA_SNOWFLAKE_PROD_NON_PII_CAMPAIGN', 'AMICA_SNOWFLAKE_PROD_PII_ANALYTICS', 'AMICA_SNOWFLAKE_UAT_PII_CAMPAIGN'
    )

Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://myapps.dentsu.com/app/snowflake/exkdty4rr8nj82z0H0i7/sso/saml?SAMLRequest=lZJBc9owEIX%2Fikc925IdJwENkKGBDExoQwP00JuwFqIgS65WjqG%2FvsKETHpIZnqTVu9J3%2Bpt72Zf6ugFHCpr%2BiRNGInAFFYqs%2B2T1fIu7pAIvTBSaGugTw6A5GbQQ1Hqig9r%2F2Qe4XcN6KNwkUHeHvRJ7Qy3AhVyI0pA7gu%2BGH6b8SxhvHLW28Jq8s7yuUMggvOB8GyRqALek%2FcVp7RpmqS5SKzb0owxRlmXBtVR8uWs34eePtCnlOVHfVAE%2BfyV7asypy%2F4DGt9EiGfLJfzeP6wWJJoeEa9tQbrEtwC3IsqYPU4OwFgIAjlnYZYlKoQCRrbbLTYQWHLqvbhwiSs6AYk1XarQs%2FTUZ9UOyWff1zk2o3Bie7Vuvuw3699I2fjfHU%2FuS9Xc3lZ201ajseqiwWJfp5DzY6hThFrmJpjlD6UWHYVszTOrpcZ46zL8yy5zNkvEo1ClMoI3zrfeA%2BiqjCRYDzWLV3Y0zdwCvud9IfcuY557mR%2F2ISpa4po6TFacpoW3hK4wX%2F8QY%2B%2BN77O3PcQw3Q0t1oVh%2BjOulL4j1NKk7StKBlvWimHUig9lNIBYkhLa9vcOhA%2BjLZ3NRA6OL3673AP%2FgI%3D&RelayState=ver%3A1

Enter the URL the SSO URL redirected you to:  http://localhost:53167/?token=7VlZl6LIEn6vX1HHfvRUsSN4umouqyKCgqDCG0uyKJssgv76i1ZbXVXd09PTM-c-3NNPmpERkZFfBOEX8vmPLk3uj6Cs4jx7GiCP8OAeZF7ux1n4NDAN8YEa_PH8uXLSBC3GOqiKPKvAPQ-qOs6c-moU1XVRjSFIAeU-AQ-MInHMY5XlbZA4e-DladH0yuFj_w0KgA8leRhng3uJfxrEPk6NMARFMRzHUQIhRjhNUAQOYwiMj3qd7HakkT8Nin3s7zQMT0oBlA5NuvSi69y69ecCbspTOTWXPtHkAZIKQkxXXm9fVQ2Qsqp2svppgMIo-QAjD-jIQOExTI9x_BEjcXtwv74BgF4A6CHJqvHLnZ8GTZmNc6eKq3HmpKAa1954xSjzca86Lsq8zr08udl01QscPRpt2z622GNehhAKwwi0VeYrLwKpM_gC5_gaXHkv5mXq1D8-5yKJ_YfgqjoGWR3Xp3dx_tjcqSpQXpI1eH4TXb6vnWtSQLf36xNellS2o9AzPIXj0WfobZDPn_1qvIrDPuVNCb4c7P_ZZWEIpqFex6_i8NPg1Rb4Uhbk1yXnZHkWe04Sn69FpIA6yv17JgnzMq6j9E9RROCL4wfQeQ8egmefBtD70H7aEYzfInxI8xJ8KivnoYoclCC_uNRBAMr-UQD3pi49DT79RalebYzSyapLkqr3y7-M5x1iIDuCJC-A_1DdrvUlpp93-B2knj8DbyxlXtJU8RGolyIpHA9U98sSBHE3j6u-CLvqVlbA-1tJgN6G92H5ggYfh33X-JsJ6hPw6V1aXrysnaQBz8aCP6UWNoOGETuvGnx4WDVUrWSkYO8brHAjij_UGDKMmv3TNaK3xlfBa5Jflh-q9LWqXiw6jV3QZ0ZNDGu32h14F9qSdHWMWmk7EltJmvucpuLaas1x

In [433]:
cur = ctx.cursor()

cur.execute('USE WAREHOUSE AMICA_WH_XSMALL') 
cur.execute("USE DATABASE FIVETRAN_AMICA_DATABASE")
cur.execute("USE SCHEMA AGGREGATE")

In [440]:

# Define your date range variables
start_date = '01-Jan-2025'
end_date = '24-jan-2026'



# Define date range variables
start_date1 = '2025-01-01'
end_date1 = '2026-01-24'



In [441]:
query = f'''


SELECT DATE, year, month, state, line, engine, keyword_type, campaign_name, campaign_type,
       SUM(impressions) AS impressions, SUM(spend) AS spend, SUM(clicks) AS clicks, SUM(conversions) AS conversions
FROM (
    SELECT
        stg_source,
        date,
        YEAR(date) AS year,
        MONTH(date) AS month,
        state_code AS state,
        campaign_name,
        campaign_type,
        CASE 
            WHEN UPPER(campaign_type) LIKE '%DEMAND_GEN' OR UPPER(campaign_type) LIKE '%VIDEO' THEN 'YouTube'
            WHEN UPPER(campaign_type) LIKE '%PERFORMANCE MAX' OR UPPER(campaign_type) LIKE '%SEARCH' THEN 'Google Search'
            ELSE campaign_type END AS engine,
        CASE
            WHEN UPPER(campaign_name) LIKE '%AUTO%' THEN 'Auto'
            WHEN UPPER(campaign_name) LIKE '%HOME%' THEN 'Home'
            WHEN UPPER(campaign_name) LIKE '%CONDO%' THEN 'Condo'
            WHEN UPPER(campaign_name) LIKE '%RENTERS%' THEN 'Renters'
            WHEN UPPER(campaign_name) LIKE '%ENTERPRISE%' OR UPPER(campaign_name) LIKE '%BRAND%' THEN 'Enterprise'
            WHEN UPPER(campaign_name) LIKE '%LIFE%' THEN 'Life'
            ELSE 'Enterprise'
        END AS line,
        CASE
            WHEN campaign_type = 'Search' THEN 
                CASE
                    WHEN UPPER(campaign_name) LIKE '%PERFORMANCE MAX%' THEN 'PMAX'
                    WHEN UPPER(campaign_name) LIKE '%BRAND%' THEN 'Brand'
                    WHEN UPPER(campaign_name) LIKE '%GENERIC%' THEN 'Generic'
                    ELSE 'None'
                END
            WHEN campaign_type = 'Performance Max' AND UPPER(campaign_name) LIKE '%PERFORMANCE MAX%' THEN 'PMAX'
            ELSE 'None'
        END AS keyword_type,
        SUM(impressions) AS impressions,
        SUM(clicks) AS clicks,
        SUM(cost) AS spend,
        SUM(conversions) AS conversions
    FROM FIVETRAN_AMICA_DATABASE.AGGREGATE.AGG_GOOGLE
    WHERE date BETWEEN '{start_date}' AND '{end_date}'
      AND engine <> 'YouTube'
      AND line <> 'Life'
      AND UPPER(campaign_name) NOT LIKE '%LIFE%'
      AND UPPER(campaign_name) NOT IN (
          'QS_AUTO_INSURANCE','MA_AUTO_INSURANCE','MA_HOME_INSURANCE','CA_AUTO_INSURANCE','USNEWS_HOME_INSURANCE',
          'CNBC','MA_HOME_INSURANCE','MA_HOME_INSURANCE','CA_HOME_INSURANCE','MONEYGROUP_TIER1PUBS',
          'BEST_MONEY_AUTO_INSURANCE','MONEY.COM','2501R','QS_AUTO_INSURANCE'
      )
    GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9, 10
)
GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9;


'''



google_data = pd.read_sql(query, ctx)
print(google_data.shape)
google_data.head()

/tmp/ipykernel_1887463/3589701959.py:62: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  google_data = pd.read_sql(query, ctx)


(70671, 13)


,DATE,YEAR,MONTH,STATE,LINE,ENGINE,KEYWORD_TYPE,CAMPAIGN_NAME,CAMPAIGN_TYPE,IMPRESSIONS,SPEND,CLICKS,CONVERSIONS
0,2025-04-01,2025,4,NH,Home,Google Search,Generic,Home Generic - NH,SEARCH,307,790.56,10,0.0
1,2025-08-12,2025,8,VA,Auto,Google Search,Generic,Auto Generic - VA,SEARCH,1392,5129.52,38,17.0
2,2025-08-19,2025,8,OR,Auto,Google Search,Brand,Amica Auto - Brand,SEARCH,32,340.46,11,5.0
3,2025-05-23,2025,5,NM,Auto,Google Search,Brand,Amica Auto - Brand,SEARCH,5,55.91,2,0.0
4,2025-03-06,2025,3,AR,Enterprise,Google Search,Brand,Amica Brand - Enterprise - Non-Strategic,SEARCH,11,18.55,4,1.0


In [442]:
google_data['Engine'] = 'Google'

In [443]:
google_data.head()

,DATE,YEAR,MONTH,STATE,LINE,ENGINE,KEYWORD_TYPE,CAMPAIGN_NAME,CAMPAIGN_TYPE,IMPRESSIONS,SPEND,CLICKS,CONVERSIONS,Engine
0,2025-04-01,2025,4,NH,Home,Google Search,Generic,Home Generic - NH,SEARCH,307,790.56,10,0.0,Google
1,2025-08-12,2025,8,VA,Auto,Google Search,Generic,Auto Generic - VA,SEARCH,1392,5129.52,38,17.0,Google
2,2025-08-19,2025,8,OR,Auto,Google Search,Brand,Amica Auto - Brand,SEARCH,32,340.46,11,5.0,Google
3,2025-05-23,2025,5,NM,Auto,Google Search,Brand,Amica Auto - Brand,SEARCH,5,55.91,2,0.0,Google
4,2025-03-06,2025,3,AR,Enterprise,Google Search,Brand,Amica Brand - Enterprise - Non-Strategic,SEARCH,11,18.55,4,1.0,Google


In [444]:
query1 = f'''


SELECT yr_mth, state, line, engine, campaign_name, keyword_type,
       SUM(impressions) AS impressions, SUM(clicks) AS clicks, SUM(spend) AS spend, SUM(conversions) AS conversions
FROM (
    SELECT 
        stg_source,
        TO_CHAR(date,'YYYY-MM') AS yr_mth,
        state_code AS state,
        'Bing' AS engine,
        NULL AS campaign_type,
        campaign_name,
        CASE
            WHEN UPPER(campaign_name) LIKE '%AUTO%' THEN 'Auto'
            WHEN UPPER(campaign_name) LIKE '%HOME%' THEN 'Home'
            WHEN UPPER(campaign_name) LIKE '%CONDO%' THEN 'Condo'
            WHEN UPPER(campaign_name) LIKE '%RENTERS%' THEN 'Renters'
            WHEN UPPER(campaign_name) LIKE '%ENTERPRISE%' THEN 'Enterprise'
            WHEN UPPER(campaign_name) LIKE '%LIFE%' THEN 'Life'
        END AS line,
        CASE 
            WHEN UPPER(campaign_name) LIKE '%BRAND%' THEN 'Brand'
            WHEN UPPER(campaign_name) LIKE '%GENERIC%' THEN 'Generic'
        END AS keyword_type,
        SUM(impressions) AS impressions,
        SUM(clicks) AS clicks,
        SUM(spend) AS spend,
        SUM(conversions) AS conversions
    FROM FIVETRAN_AMICA_DATABASE.AGGREGATE.AGG_BING
    WHERE date BETWEEN '{start_date}' AND '{end_date}'
      AND line <> 'Life'
      AND UPPER(campaign_name) NOT LIKE '%LIFE%'
      AND UPPER(campaign_name) NOT IN (
          'QS_AUTO_INSURANCE','MA_AUTO_INSURANCE','MA_HOME_INSURANCE','CA_AUTO_INSURANCE','USNEWS_HOME_INSURANCE',
          'CNBC','MA_HOME_INSURANCE','MA_HOME_INSURANCE','CA_HOME_INSURANCE','MONEYGROUP_TIER1PUBS',
          'BEST_MONEY_AUTO_INSURANCE','MONEY.COM','2502C','2501R','QS_AUTO_INSURANCE'
      )
    GROUP BY 1, 2, 3, 4, 5, 6, 7, 8
)
GROUP BY 1, 2, 3, 4, 5, 6;

'''

bing_data = pd.read_sql(query1, ctx)
print(bing_data.shape)
bing_data.head()

/tmp/ipykernel_1887463/1293090693.py:45: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  bing_data = pd.read_sql(query1, ctx)


(2680, 10)


,YR_MTH,STATE,LINE,ENGINE,CAMPAIGN_NAME,KEYWORD_TYPE,IMPRESSIONS,CLICKS,SPEND,CONVERSIONS
0,2025-11,TN,Enterprise,Bing,Amica Brand - Enterprise - Non Strategic - Bing,Brand,210,57,633.08,14
1,2025-10,AZ,Auto,Bing,Auto Generic - AZ-Experiment - VBB,Generic,7637,247,20840.97,11
2,2025-04,AZ,Enterprise,Bing,Amica Brand - Enterprise - Strategic - Bing,Brand,534,160,1257.87,41
3,2025-02,IN,Enterprise,Bing,Amica Brand - Enterprise - Non Strategic - Bin...,Brand,118,29,288.19,7
4,2025-09,MA,Auto,Bing,Auto Generic - MA,Generic,24772,812,87784.10,196


In [445]:
bing_data['Engine'] = 'Bing'

In [446]:
### internal_data

In [604]:
query2 = f'''

select a.lead_storage_nbr, a.line_of_business,a.new_segment_reporting_cd as quality_seg,
a.lead_dt, a.lead_contact_method,
a.lead_medium,b.medium_channel,b.medium,b.source,b.campaign,
sum(a.lead)lead,sum(quote)quote,sum(a.issued_policy)issue,
sum(case when lead_dt=current_proc_dt then issued_policy else 0 end ) Day0,
sum(case when current_proc_dt - lead_dt <=6 then issued_policy else 0 end ) Day7,
state_cd,
case when state_cd in('CT','ME','NY','PA','RI','VT','NH','MA') then 'Target'
     when state_cd in('AZ','DC','DE','MD','NC','NM','OR','VA','WA') then 'Tier2'
else 'Non-Target' 
end state_grp,
to_char(lead_dt,'YYYY-MM') lead_monyr
 
from  PROD_DB_PII_ANALYTICS.SANDBOX.leads_policies_auto_home_pup  a,
PROD_DB_PII_ANALYTICS.SANDBOX.last_touch_attribution_vq b
where a.lead_Dt between'{start_date}' AND '{end_date}'
and b.medium in('cpc','Paid Search')
and a.lead_storage_nbr=b.orig_storage_no
group by 1,2,3,4,5,6,7,8,9,10,16,17,18;

'''

internal_data = pd.read_sql(query2, ctx)
print(internal_data.shape)
internal_data.head()

/tmp/ipykernel_1887463/1854016874.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  internal_data = pd.read_sql(query2, ctx)


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://myapps.dentsu.com/app/snowflake/exkdty4rr8nj82z0H0i7/sso/saml?SAMLRequest=lZJRb9owFIX%2FSuQ9JzYpbcECKgbrQOoYLbBJe3OSC3Xj2JmvQ8h%2B%2FUwoVfvQSn2zr8%2Bxv%2BtzBzeHQgV7sCiNHpJOxEgAOjWZ1Lsh2axvwx4J0AmdCWU0DEkDSG5GAxSFKvm4co%2F6Af5WgC7wF2nk7cGQVFZzI1Ai16IA5C7lq%2FGPOx5HjJfWOJMaRV5ZPnYIRLDOE54tGUqP9%2BhcySmt6zqqLyJjdzRmjFHWp151lHw56w%2B%2Bp3f0Hcq6R71XePnyme2r1Kcv%2BAgrOYmQz9brZbj8uVqTYHxGnRiNVQF2BXYvU9g83J0A0BP4cq4gFIVMRYTa1FslckhNUVbOXxj5Fd1CRpXZSd%2FzfDokZS6z68nmapcsxjFLft8%2FybycNXq5%2Bp6UMq%2BK%2FaLJ75ODwPQbJLuUBL%2FOocbHUOeIFcz1MUrnSyy%2BClknjPtrdsHjHu9eRv3L%2Fh8STH2UUgvXOl94G1GWGGWgHVYtnd%2FTF3AKhzxzTdfann7qxf%2FYjMlrimjoMVpymhbeEtjRJ%2F5gQF8bn2du4WOYT5dGybQJbo0thHs%2FpU7UaSsyC7etlEMhpBpnmQVEn5ZSpp5YEM6PtrMVEDo6vfp2uEf%2FAQ%3D%3D&Rela

Enter the URL the SSO URL redirected you to:  http://localhost:49007/?token=7VlZl5raEn7vX9HLPLo6DIKAK93nMioiKJMKbwwbRJlkEOTXX7Rjp7uTk-QkZ92Hu_Kku3ZV7dpfFcVX-umvNonvT6Aooyx9HCAf4cE9SL3Mj9LwcWAawgM5-OvpU-kkMZpPNFDmWVqCew6UVZQ61dVoV1V5OYEgGRSHGDzQssjSH8s0a4LYOQAvS_K6Vw4_9t-gAPhQnIVROrgXucdB5D8Q8IigcIIiEQQbY9gIQ0gU67-j-KjXSW9HGtnjID9EPsGa49BVaBR2N-o-OuSzc7rSp24eHerkpJwPqts6pccDN_R6-7KsgZiWlZNWjwMURscPMPKAUgY8mqDUBEU-khhiD-7XNwDQCwA9JGk5eb7z46Au0knmlFE5SZ0ElJPKm-i0vJj0qpO8yKrMy-KbTVs-w9Gj0TTNx2b0MStCCIVhBNrKC93bgcQZfIZzcg2uuBeyInGq759zkfRYBVfVCUirqDq_ifP75k5ZguKSrMHTq-iyQ-VckwLag1-dsaIg0z2JdvAMjohP0Osgnz755USPwj7ldQE-H-z_3WVhCKagXscvo_DD4MUW-GIaZNcl66RZGnlOHHXXIpJBtcv8ezoOsyKqdsnfoojAF8cPoPUePARLPwygt6H9tCMYu0X4kGQF-FCUzkO5c1B8_NmlBgJQ9I8CuDc18XHw4QelerUxCictL0kq3y5_GM8bxEB6AnGWA_-hvF3rc0w_7_AbSD19At5ETL24LqMTUC5FkjseKO9XBQiidhGVfRG25a2sgPePkgC9Du_d8hkNLgr7rvEPE9Qn4MObtDx7WTtxDZ4Chl2fqGxrcRHQZ3RqVoSDFccVwXYJnU1n2mnUwk7GCXXzeI3otfFV8JLk5-W7Kn2pqmcLRoKiYDtUnNO8Y1trtCUNQCmGQrXq1FrEnC6teFxv9L3o

(325407, 18)


,LEAD_STORAGE_NBR,LINE_OF_BUSINESS,QUALITY_SEG,LEAD_DT,LEAD_CONTACT_METHOD,LEAD_MEDIUM,MEDIUM_CHANNEL,MEDIUM,SOURCE,CAMPAIGN,LEAD,QUOTE,ISSUE,DAY0,DAY7,STATE_CD,STATE_GRP,LEAD_MONYR
0,Q007271432,AUTOMOBILE,0,2025-02-25,PHONE,Paid Search,Paid Search,Paid Search,None,Home Call Extension - Conversion Tracking,1,0,0,0,0,CT,Target,2025-02
1,W011718556,AUTOMOBILE,0,2025-05-13,WEB,Paid Search,Paid Search,cpc,google,auto generic - dc,0,0,0,0,0,MD,Tier2,2025-05
2,W011537807,AUTOMOBILE,1,2025-03-27,WEB,Paid Search,Paid Search,cpc,bing,enterprise - strategic - bing,1,1,0,0,0,MA,Target,2025-03
3,Q008673116,AUTOMOBILE,2,2025-11-02,PHONE,Paid Search,Paid Search,Paid Search,None,Pmax Test Pilot,1,1,0,0,0,MA,Target,2025-11
4,W011298556,AUTOMOBILE,5,2025-01-29,WEB,Paid Search,Paid Search,cpc,google,auto generic - az gbb,1,1,0,0,0,AZ,Tier2,2025-01


In [448]:
internal_data.columns = internal_data.columns.str.lower()

In [605]:
internal_data.head()

,LEAD_STORAGE_NBR,LINE_OF_BUSINESS,QUALITY_SEG,LEAD_DT,LEAD_CONTACT_METHOD,LEAD_MEDIUM,MEDIUM_CHANNEL,MEDIUM,SOURCE,CAMPAIGN,LEAD,QUOTE,ISSUE,DAY0,DAY7,STATE_CD,STATE_GRP,LEAD_MONYR
0,Q007271432,AUTOMOBILE,0,2025-02-25,PHONE,Paid Search,Paid Search,Paid Search,None,Home Call Extension - Conversion Tracking,1,0,0,0,0,CT,Target,2025-02
1,W011718556,AUTOMOBILE,0,2025-05-13,WEB,Paid Search,Paid Search,cpc,google,auto generic - dc,0,0,0,0,0,MD,Tier2,2025-05
2,W011537807,AUTOMOBILE,1,2025-03-27,WEB,Paid Search,Paid Search,cpc,bing,enterprise - strategic - bing,1,1,0,0,0,MA,Target,2025-03
3,Q008673116,AUTOMOBILE,2,2025-11-02,PHONE,Paid Search,Paid Search,Paid Search,None,Pmax Test Pilot,1,1,0,0,0,MA,Target,2025-11
4,W011298556,AUTOMOBILE,5,2025-01-29,WEB,Paid Search,Paid Search,cpc,google,auto generic - az gbb,1,1,0,0,0,AZ,Tier2,2025-01


In [450]:
#internal_data1 = pd.read_csv('/data/Projects/lead_policy_counts/aggregated_df_search.csv')

In [451]:
exclude_list = [
    'QS_AUTO_INSURANCE', 'MA_AUTO_INSURANCE', 'MA_HOME_INSURANCE', 'CA_AUTO_INSURANCE',
    'USNEWS_HOME_INSURANCE', 'CNBC', 'CA_HOME_INSURANCE', 'MONEYGROUP_TIER1PUBS',
    'BEST_MONEY_AUTO_INSURANCE', 'MONEY.COM', '2501R'
]

internal_data = internal_data[
    ~internal_data['campaign'].str.upper().isin(exclude_list)
]

In [452]:
internal_data.lead.sum()

315978

In [453]:
internal_data.issue.sum()

15203

In [454]:
pd.set_option('display.max_columns', None)

### Web Leads

In [463]:
internal_data = internal_data[internal_data['lead_contact_method'] == 'WEB']

In [464]:
google_data.head(3)

,DATE,YEAR,MONTH,STATE,LINE,ENGINE,KEYWORD_TYPE,CAMPAIGN_NAME,CAMPAIGN_TYPE,IMPRESSIONS,SPEND,CLICKS,CONVERSIONS,Engine
0,2025-04-01,2025,4,NH,Home,Google Search,Generic,Home Generic - NH,SEARCH,307,790.56,10,0.0,Google
1,2025-08-12,2025,8,VA,Auto,Google Search,Generic,Auto Generic - VA,SEARCH,1392,5129.52,38,17.0,Google
2,2025-08-19,2025,8,OR,Auto,Google Search,Brand,Amica Auto - Brand,SEARCH,32,340.46,11,5.0,Google


In [465]:
internal_data.head(3)

,lead_storage_nbr,line_of_business,quality_seg,lead_dt,lead_contact_method,lead_medium,medium_channel,medium,source,campaign,lead,quote,issue,day0,day7,state_cd,state_grp,lead_monyr
1,W012406931,AUTOMOBILE,4,2025-11-03,WEB,Paid Search,Paid Search,cpc,google,auto generic wa performance max,1,1,0,0,0,WA,Tier2,2025-11
2,W012331841,AUTOMOBILE,2,2025-10-14,WEB,Paid Search,Paid Search,cpc,google,auto generic nh performance max,1,1,0,0,0,VA,Tier2,2025-10
3,W011871040,AUTOMOBILE,1,2025-06-23,WEB,Paid Search,Paid Search,cpc,google,enterprise - non - strategic,1,1,0,0,0,OH,Non-Target,2025-06


In [466]:

internal_data['Engine'] = internal_data.apply(
    lambda row: (
        'bing'
        # safe check: convert campaign to string, then lowercase
        if 'bing' in str(row['campaign']).lower()
        else row['source'] if row['source'] in ['google', 'bing']
        else 'google' if row['source'] == 'google.com'
        else 'bing' if row['source'] == 'bing.com'
        else 'google'
    ),
    axis=1
)


In [467]:
type(internal_data['lead_dt'][1])

datetime.date

In [468]:
start_date1 = pd.to_datetime(start_date1)
end_date1 = pd.to_datetime(end_date1)
internal_data['lead_dt'] = pd.to_datetime(internal_data['lead_dt'])
internal_data = internal_data[(internal_data['lead_dt'] >= start_date1) &(internal_data['lead_dt'] <= end_date1)]
internal_data = internal_data[internal_data['medium'].isin(['Paid Search', 'cpc'])]

In [469]:
internal_data['Engine'].value_counts()

Engine
google    226521
bing       54255
Name: count, dtype: int64

In [470]:
google_data['DATE'] = pd.to_datetime(google_data['DATE'])
google_data['lead_monyr'] = google_data['DATE'].dt.strftime('%Y-%m')

In [471]:
google_data.head(1)

,DATE,YEAR,MONTH,STATE,LINE,ENGINE,KEYWORD_TYPE,CAMPAIGN_NAME,CAMPAIGN_TYPE,IMPRESSIONS,SPEND,CLICKS,CONVERSIONS,Engine,lead_monyr
0,2025-04-01,2025,4,NH,Home,Google Search,Generic,Home Generic - NH,SEARCH,307,790.56,10,0.0,Google,2025-04


In [472]:
bing_data.head(1)

,YR_MTH,STATE,LINE,ENGINE,CAMPAIGN_NAME,KEYWORD_TYPE,IMPRESSIONS,CLICKS,SPEND,CONVERSIONS,Engine
0,2025-11,TN,Enterprise,Bing,Amica Brand - Enterprise - Non Strategic - Bing,Brand,210,57,633.08,14,Bing


In [473]:
bing_data.rename(columns={'YR_MTH': 'lead_monyr'}, inplace=True)

In [474]:
internal_data['lead_dt'] = pd.to_datetime(internal_data['lead_dt'])
internal_data['lead_monyr'] = internal_data['lead_dt'].dt.strftime('%Y-%m')

In [475]:
lookup_df = pd.read_excel('/data/Users/Shelly/Search_Report/Search_camp_lookup.xlsx')

In [476]:
lookup_df.head()

,int_names,eng_names
0,AUTO GENERIC WA PERFORMANCE MAX,AUTO GENERIC - WA - PERFORMANCE MAX
1,AUTO - BRAND,AMICA AUTO - BRAND
2,ENTERPRISE - STRATEGIC - BING,AMICA BRAND - ENTERPRISE - STRATEGIC - BING
3,ENTERPRISE - STRATEGIC,AMICA BRAND - ENTERPRISE - STRATEGIC
4,ENTERPRISE - NON - STRATEGIC,AMICA BRAND - ENTERPRISE - NON-STRATEGIC


In [477]:
internal_data['campaign'] = internal_data['campaign'].astype(str).str.upper()

In [478]:
internal_data.shape

(280776, 19)

In [479]:
internal_data.head()

,lead_storage_nbr,line_of_business,quality_seg,lead_dt,lead_contact_method,lead_medium,medium_channel,medium,source,campaign,lead,quote,issue,day0,day7,state_cd,state_grp,lead_monyr,Engine
1,W012406931,AUTOMOBILE,4,2025-11-03,WEB,Paid Search,Paid Search,cpc,google,AUTO GENERIC WA PERFORMANCE MAX,1,1,0,0,0,WA,Tier2,2025-11,google
2,W012331841,AUTOMOBILE,2,2025-10-14,WEB,Paid Search,Paid Search,cpc,google,AUTO GENERIC NH PERFORMANCE MAX,1,1,0,0,0,VA,Tier2,2025-10,google
3,W011871040,AUTOMOBILE,1,2025-06-23,WEB,Paid Search,Paid Search,cpc,google,ENTERPRISE - NON - STRATEGIC,1,1,0,0,0,OH,Non-Target,2025-06,google
4,W012196280,AUTOMOBILE,1,2025-09-11,WEB,Paid Search,Paid Search,cpc,google,HOME GENERIC RI PERFORMANCE MAX,1,1,0,0,0,MA,Target,2025-09,google
5,W011310212,AUTOMOBILE,4,2025-02-01,WEB,Paid Search,Paid Search,cpc,bing,AUTO GENERIC - AZ,1,1,0,0,0,AZ,Tier2,2025-02,bing


In [480]:
def classify_keyword(campaign):
    campaign_upper = str(campaign).upper()
    if 'BRAND' in campaign_upper or 'ENTERPRISE' in campaign_upper:
        return 'Brand'
    elif 'MAX' in campaign_upper:
        return 'PMAX'
    else:
        return 'Generic'

internal_data['keyword_type'] = internal_data['campaign'].apply(classify_keyword)

In [481]:
google_data['keyword_type'] = google_data['CAMPAIGN_NAME'].apply(classify_keyword)

In [482]:
bing_data['keyword_type'] = bing_data['CAMPAIGN_NAME'].apply(classify_keyword)

In [483]:
internal_data.head()

,lead_storage_nbr,line_of_business,quality_seg,lead_dt,lead_contact_method,lead_medium,medium_channel,medium,source,campaign,lead,quote,issue,day0,day7,state_cd,state_grp,lead_monyr,Engine,keyword_type
1,W012406931,AUTOMOBILE,4,2025-11-03,WEB,Paid Search,Paid Search,cpc,google,AUTO GENERIC WA PERFORMANCE MAX,1,1,0,0,0,WA,Tier2,2025-11,google,PMAX
2,W012331841,AUTOMOBILE,2,2025-10-14,WEB,Paid Search,Paid Search,cpc,google,AUTO GENERIC NH PERFORMANCE MAX,1,1,0,0,0,VA,Tier2,2025-10,google,PMAX
3,W011871040,AUTOMOBILE,1,2025-06-23,WEB,Paid Search,Paid Search,cpc,google,ENTERPRISE - NON - STRATEGIC,1,1,0,0,0,OH,Non-Target,2025-06,google,Brand
4,W012196280,AUTOMOBILE,1,2025-09-11,WEB,Paid Search,Paid Search,cpc,google,HOME GENERIC RI PERFORMANCE MAX,1,1,0,0,0,MA,Target,2025-09,google,PMAX
5,W011310212,AUTOMOBILE,4,2025-02-01,WEB,Paid Search,Paid Search,cpc,bing,AUTO GENERIC - AZ,1,1,0,0,0,AZ,Tier2,2025-02,bing,Generic


In [484]:
internal_data['Auto_Phone_leads'] = internal_data.apply(lambda row: row['lead'] if row['line_of_business'] == 'AUTOMOBILE' and row['lead_contact_method'] == 'PHONE' else None,axis=1)
internal_data['Auto_Web_leads'] = internal_data.apply(lambda row: row['lead'] if row['line_of_business'] == 'AUTOMOBILE' and row['lead_contact_method'] == 'WEB' else None,axis=1)
internal_data['Seg_1_2_Auto_Web_leads'] = internal_data.apply(lambda row: row['Auto_Web_leads'] if row['quality_seg'] in [1, 2] else 0,axis=1)
internal_data['Seg_0_Auto_Web_leads'] = internal_data.apply(lambda row: 1 if (row['line_of_business'] == 'AUTOMOBILE' and row['quality_seg'] == 0 and row['lead_contact_method'] == 'WEB') else 0,axis=1)
internal_data['Seg_1_2_Auto_Phone_leads'] = internal_data.apply(lambda row: row['Auto_Phone_leads'] if row['quality_seg'] in [1, 2] else 0,axis=1)
internal_data['Seg_0_Auto_Phone_leads'] = internal_data.apply(lambda row: 1 if (row['line_of_business'] == 'AUTOMOBILE' and row['quality_seg'] == 0 and row['lead_contact_method'] == 'PHONE') else 0,axis=1)
internal_data['Auto_Issues'] = internal_data.apply(lambda row: row['issue'] if row['line_of_business'] == 'AUTOMOBILE' else None,axis=1)
internal_data['Auto_Web_Issues'] = internal_data.apply(lambda row: row['issue'] if (row['line_of_business'] == 'AUTOMOBILE' and row['lead_contact_method'] == 'WEB') else None, axis=1)
internal_data['Auto_Phone_Issues'] = internal_data.apply(lambda row: row['issue'] if (row['line_of_business'] == 'AUTOMOBILE' and row['lead_contact_method'] == 'PHONE') else None, axis=1)
internal_data['Auto_Day7'] = internal_data.apply(lambda row: row['day7'] if row['line_of_business'] == 'AUTOMOBILE' else None,axis=1)
internal_data['Auto_Web_Day7'] = internal_data.apply(lambda row: row['day7'] if (row['line_of_business'] == 'AUTOMOBILE' and row['lead_contact_method'] == 'WEB') else None,axis=1)
internal_data['Auto_Phone_Day7'] = internal_data.apply(lambda row: row['day7'] if (row['line_of_business'] == 'AUTOMOBILE' and row['lead_contact_method'] == 'PHONE') else None,axis=1)

In [485]:
internal_data['Home_Phone_leads'] = internal_data.apply(lambda row: row['lead'] if row['line_of_business'] == 'HOMEOWNERS' and row['lead_contact_method'] == 'PHONE' else None,axis=1)
internal_data['Home_Web_leads'] = internal_data.apply(lambda row: row['lead'] if row['line_of_business'] == 'HOMEOWNERS' and row['lead_contact_method'] == 'WEB' else None,axis=1)
internal_data['Home_Issues'] = internal_data.apply(lambda row: row['issue'] if row['line_of_business'] == 'HOMEOWNERS' else None,axis=1)
internal_data['Home_Web_Issues'] = internal_data.apply(lambda row: row['issue'] if (row['line_of_business'] == 'HOMEOWNERS' and row['lead_contact_method'] == 'WEB') else None,axis=1)
internal_data['Home_Phone_Issues'] = internal_data.apply(lambda row: row['issue'] if (row['line_of_business'] == 'HOMEOWNERS' and row['lead_contact_method'] == 'PHONE') else None,axis=1)
internal_data['Home_Day7'] = internal_data.apply(lambda row: row['day7'] if row['line_of_business'] == 'HOMEOWNERS' else None,axis=1)
internal_data['Home_Web_Day7'] = internal_data.apply(lambda row: row['day7'] if (row['line_of_business'] == 'HOMEOWNERS' and row['lead_contact_method'] == 'WEB') else None,axis=1)
internal_data['Home_Web_Day7'] = internal_data.apply(lambda row: row['day7'] if (row['line_of_business'] == 'HOMEOWNERS' and row['lead_contact_method'] == 'PHONE') else None,axis=1)

In [486]:
internal_data.head()

,lead_storage_nbr,line_of_business,quality_seg,lead_dt,lead_contact_method,lead_medium,medium_channel,medium,source,campaign,lead,quote,issue,day0,day7,state_cd,state_grp,lead_monyr,Engine,keyword_type,Auto_Phone_leads,Auto_Web_leads,Seg_1_2_Auto_Web_leads,Seg_0_Auto_Web_leads,Seg_1_2_Auto_Phone_leads,Seg_0_Auto_Phone_leads,Auto_Issues,Auto_Web_Issues,Auto_Phone_Issues,Auto_Day7,Auto_Web_Day7,Auto_Phone_Day7,Home_Phone_leads,Home_Web_leads,Home_Issues,Home_Web_Issues,Home_Phone_Issues,Home_Day7,Home_Web_Day7
1,W012406931,AUTOMOBILE,4,2025-11-03,WEB,Paid Search,Paid Search,cpc,google,AUTO GENERIC WA PERFORMANCE MAX,1,1,0,0,0,WA,Tier2,2025-11,google,PMAX,None,1.0,0.0,0,0.0,0,0.0,0.0,None,0.0,0.0,None,None,NaN,NaN,NaN,None,NaN,None
2,W012331841,AUTOMOBILE,2,2025-10-14,WEB,Paid Search,Paid Search,cpc,google,AUTO GENERIC NH PERFORMANCE MAX,1,1,0,0,0,VA,Tier2,2025-10,google,PMAX,None,1.0,1.0,0,NaN,0,0.0,0.0,None,0.0,0.0,None,None,NaN,NaN,NaN,None,NaN,None
3,W011871040,AUTOMOBILE,1,2025-06-23,WEB,Paid Search,Paid Search,cpc,google,ENTERPRISE - NON - STRATEGIC,1,1,0,0,0,OH,Non-Target,2025-06,google,Brand,None,1.0,1.0,0,NaN,0,0.0,0.0,None,0.0,0.0,None,None,NaN,NaN,NaN,None,NaN,None
4,W012196280,AUTOMOBILE,1,2025-09-11,WEB,Paid Search,Paid Search,cpc,google,HOME GENERIC RI PERFORMANCE MAX,1,1,0,0,0,MA,Target,2025-09,google,PMAX,None,1.0,1.0,0,NaN,0,0.0,0.0,None,0.0,0.0,None,None,NaN,NaN,NaN,None,NaN,None
5,W011310212,AUTOMOBILE,4,2025-02-01,WEB,Paid Search,Paid Search,cpc,bing,AUTO GENERIC - AZ,1,1,0,0,0,AZ,Tier2,2025-02,bing,Generic,None,1.0,0.0,0,0.0,0,0.0,0.0,None,0.0,0.0,None,None,NaN,NaN,NaN,None,NaN,None


In [487]:
# Create a dictionary from the lookup table
lookup_dict = dict(zip(lookup_df['int_names'], lookup_df['eng_names']))

# Replace campaign names using the dictionary
internal_data['campaign'] = internal_data['campaign'].apply(lambda x: lookup_dict.get(x, x))

In [488]:
internal_data.head()

,lead_storage_nbr,line_of_business,quality_seg,lead_dt,lead_contact_method,lead_medium,medium_channel,medium,source,campaign,lead,quote,issue,day0,day7,state_cd,state_grp,lead_monyr,Engine,keyword_type,Auto_Phone_leads,Auto_Web_leads,Seg_1_2_Auto_Web_leads,Seg_0_Auto_Web_leads,Seg_1_2_Auto_Phone_leads,Seg_0_Auto_Phone_leads,Auto_Issues,Auto_Web_Issues,Auto_Phone_Issues,Auto_Day7,Auto_Web_Day7,Auto_Phone_Day7,Home_Phone_leads,Home_Web_leads,Home_Issues,Home_Web_Issues,Home_Phone_Issues,Home_Day7,Home_Web_Day7
1,W012406931,AUTOMOBILE,4,2025-11-03,WEB,Paid Search,Paid Search,cpc,google,AUTO GENERIC - WA - PERFORMANCE MAX,1,1,0,0,0,WA,Tier2,2025-11,google,PMAX,None,1.0,0.0,0,0.0,0,0.0,0.0,None,0.0,0.0,None,None,NaN,NaN,NaN,None,NaN,None
2,W012331841,AUTOMOBILE,2,2025-10-14,WEB,Paid Search,Paid Search,cpc,google,AUTO GENERIC - NH - PERFORMANCE MAX,1,1,0,0,0,VA,Tier2,2025-10,google,PMAX,None,1.0,1.0,0,NaN,0,0.0,0.0,None,0.0,0.0,None,None,NaN,NaN,NaN,None,NaN,None
3,W011871040,AUTOMOBILE,1,2025-06-23,WEB,Paid Search,Paid Search,cpc,google,AMICA BRAND - ENTERPRISE - NON-STRATEGIC,1,1,0,0,0,OH,Non-Target,2025-06,google,Brand,None,1.0,1.0,0,NaN,0,0.0,0.0,None,0.0,0.0,None,None,NaN,NaN,NaN,None,NaN,None
4,W012196280,AUTOMOBILE,1,2025-09-11,WEB,Paid Search,Paid Search,cpc,google,HOME GENERIC - RI - PERFORMANCE MAX,1,1,0,0,0,MA,Target,2025-09,google,PMAX,None,1.0,1.0,0,NaN,0,0.0,0.0,None,0.0,0.0,None,None,NaN,NaN,NaN,None,NaN,None
5,W011310212,AUTOMOBILE,4,2025-02-01,WEB,Paid Search,Paid Search,cpc,bing,AUTO GENERIC - AZ,1,1,0,0,0,AZ,Tier2,2025-02,bing,Generic,None,1.0,0.0,0,0.0,0,0.0,0.0,None,0.0,0.0,None,None,NaN,NaN,NaN,None,NaN,None


In [489]:
bing_data.head()

,lead_monyr,STATE,LINE,ENGINE,CAMPAIGN_NAME,KEYWORD_TYPE,IMPRESSIONS,CLICKS,SPEND,CONVERSIONS,Engine,keyword_type
0,2025-11,TN,Enterprise,Bing,Amica Brand - Enterprise - Non Strategic - Bing,Brand,210,57,633.08,14,Bing,Brand
1,2025-10,AZ,Auto,Bing,Auto Generic - AZ-Experiment - VBB,Generic,7637,247,20840.97,11,Bing,Generic
2,2025-04,AZ,Enterprise,Bing,Amica Brand - Enterprise - Strategic - Bing,Brand,534,160,1257.87,41,Bing,Brand
3,2025-02,IN,Enterprise,Bing,Amica Brand - Enterprise - Non Strategic - Bin...,Brand,118,29,288.19,7,Bing,Brand
4,2025-09,MA,Auto,Bing,Auto Generic - MA,Generic,24772,812,87784.10,196,Bing,Generic


In [490]:
google_data['Engine'] = google_data['Engine'].astype(str).str.upper()
bing_data['Engine'] = bing_data['Engine'].astype(str).str.upper()

In [491]:
# Group Google Ads data
google_grouped = google_data.groupby(['lead_monyr', 'CAMPAIGN_NAME','keyword_type','Engine'], as_index=False).agg({
    'IMPRESSIONS': 'sum',
    'SPEND': 'sum',
    'CLICKS': 'sum',
    'CONVERSIONS': 'sum'
})

# Group Google Ads data
bing_grouped = bing_data.groupby(['lead_monyr', 'CAMPAIGN_NAME','keyword_type','Engine'], as_index=False).agg({
    'IMPRESSIONS': 'sum',
    'SPEND': 'sum',
    'CLICKS': 'sum',
    'CONVERSIONS': 'sum'
})

# Group Internal Company data
internal_grouped = internal_data.groupby(['lead_monyr', 'campaign','keyword_type','Engine'], as_index=False).agg({
    'lead': 'sum',
    'quote': 'sum',
    'issue': 'sum',
    'day0': 'sum',
    'day7': 'sum',
    'Auto_Phone_leads' :'sum',
    'Auto_Web_leads':'sum',
    'Seg_1_2_Auto_Web_leads':'sum',
    'Seg_0_Auto_Web_leads':'sum',
    'Seg_1_2_Auto_Phone_leads':'sum',
    'Seg_0_Auto_Phone_leads':'sum',
    'Auto_Issues':'sum',
    'Home_Phone_leads':'sum',
    'Home_Web_leads':'sum',
    'Home_Issues':'sum',
    'Auto_Day7':'sum',
    'Home_Day7':'sum',
    'Auto_Web_Issues' :'sum',
    'Auto_Phone_Issues' :'sum',
    'Auto_Web_Day7' :'sum',
    'Auto_Web_Day7' :'sum',
    'Home_Web_Issues' :'sum',
    'Home_Phone_Issues' :'sum',
    'Home_Web_Day7' :'sum',
    'Home_Web_Day7' :'sum'
    
})

In [492]:
engine = pd.concat([google_grouped, bing_grouped], axis=0, ignore_index=True)

In [493]:
engine.head()

,lead_monyr,CAMPAIGN_NAME,keyword_type,Engine,IMPRESSIONS,SPEND,CLICKS,CONVERSIONS
0,2025-01,Amica Auto - Brand,Brand,GOOGLE,14789,145271.48,5629,1584.51
1,2025-01,Amica Brand - Condo Insurance,Brand,GOOGLE,83,1025.64,33,8.04
2,2025-01,Amica Brand - Enterprise - Non-Strategic,Brand,GOOGLE,13130,42192.42,5008,1478.36
3,2025-01,Amica Brand - Enterprise - Strategic,Brand,GOOGLE,38765,114871.51,17381,4825.56
4,2025-01,Amica Brand - Home Insurance,Brand,GOOGLE,8875,88551.02,2974,1147.42


In [494]:
lookup_df1 = pd.read_excel('/data/Users/Shelly/Search_Report/Search_camp_lookup1.xlsx')

In [495]:
lookup_df1.head()

,eng_names,int_names
0,AMICA BRAND - ENTERPRISE - NON STRATEGIC - BIN...,AMICA BRAND - ENTERPRISE - NON STRATEGIC - BING
1,AMICA BRAND - ENTERPRISE - STRATEGIC - BING-EX...,AMICA BRAND - ENTERPRISE - STRATEGIC - BING
2,AMICA BRAND - ENTERPRISE - NON-STRATEGIC_ADAI,AMICA BRAND - ENTERPRISE - NON STRATEGIC - BING
3,AMICA BRAND - ENTERPRISE - STRATEGIC_ADAI,AMICA BRAND - ENTERPRISE - STRATEGIC - BING
4,AUTO GENERIC - MA | LP EXPERIMENT,AUTO GENERIC - MA


In [496]:
# Convert campaign names to uppercase
engine['CAMPAIGN_NAME'] = engine['CAMPAIGN_NAME'].str.upper()
engine['Engine'] = engine['Engine'].str.upper()
internal_grouped['campaign'] = internal_grouped['campaign'].str.upper()
internal_grouped['Engine'] = internal_grouped['Engine'].str.upper()

In [497]:
# Create a dictionary from the lookup table
lookup_dict1 = dict(zip(lookup_df1['eng_names'], lookup_df1['int_names']))

# Replace campaign names using the dictionary
engine['CAMPAIGN_NAME'] = engine['CAMPAIGN_NAME'].apply(lambda x: lookup_dict1.get(x, x))

In [498]:
internal_grouped.head()

,lead_monyr,campaign,keyword_type,Engine,lead,quote,issue,day0,day7,Auto_Phone_leads,Auto_Web_leads,Seg_1_2_Auto_Web_leads,Seg_0_Auto_Web_leads,Seg_1_2_Auto_Phone_leads,Seg_0_Auto_Phone_leads,Auto_Issues,Home_Phone_leads,Home_Web_leads,Home_Issues,Auto_Day7,Home_Day7,Auto_Web_Issues,Auto_Phone_Issues,Auto_Web_Day7,Home_Web_Issues,Home_Phone_Issues,Home_Web_Day7
0,2025-01,AMICA AUTO - BRAND,Brand,GOOGLE,2004,2004,59,12,33,0,1518.0,728.0,3,0.0,0,46.0,0,486.0,13.0,27.0,6.0,46.0,0,27.0,13.0,0,0
1,2025-01,AMICA AUTO - BRAND - BING,Brand,BING,370,370,24,2,8,0,278.0,149.0,2,0.0,0,18.0,0,92.0,6.0,7.0,1.0,18.0,0,7.0,6.0,0,0
2,2025-01,AMICA BRAND - CONDO INSURANCE,Brand,GOOGLE,11,11,0,0,0,0,3.0,3.0,0,0.0,0,0.0,0,8.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0
3,2025-01,AMICA BRAND - ENTERPRISE - NON STRATEGIC - BING,Brand,BING,532,532,8,0,3,0,311.0,223.0,0,0.0,0,4.0,0,221.0,4.0,1.0,2.0,4.0,0,1.0,4.0,0,0
4,2025-01,AMICA BRAND - ENTERPRISE - NON-STRATEGIC,Brand,GOOGLE,1614,1614,26,2,5,0,903.0,641.0,2,0.0,0,16.0,0,711.0,10.0,5.0,0.0,16.0,0,5.0,10.0,0,0


In [499]:
engine.head()

,lead_monyr,CAMPAIGN_NAME,keyword_type,Engine,IMPRESSIONS,SPEND,CLICKS,CONVERSIONS
0,2025-01,AMICA AUTO - BRAND,Brand,GOOGLE,14789,145271.48,5629,1584.51
1,2025-01,AMICA BRAND - CONDO INSURANCE,Brand,GOOGLE,83,1025.64,33,8.04
2,2025-01,AMICA BRAND - ENTERPRISE - NON-STRATEGIC,Brand,GOOGLE,13130,42192.42,5008,1478.36
3,2025-01,AMICA BRAND - ENTERPRISE - STRATEGIC,Brand,GOOGLE,38765,114871.51,17381,4825.56
4,2025-01,AMICA BRAND - HOME INSURANCE,Brand,GOOGLE,8875,88551.02,2974,1147.42


In [500]:
engine = engine.groupby(['lead_monyr', 'CAMPAIGN_NAME','keyword_type','Engine'], as_index=False).agg({
     'IMPRESSIONS': 'sum',
    'SPEND': 'sum',
    'CLICKS': 'sum',
    'CONVERSIONS': 'sum'
})

In [501]:
engine.shape

(1242, 8)

In [502]:
internal_grouped.shape

(1343, 27)

In [503]:
engine.head()

,lead_monyr,CAMPAIGN_NAME,keyword_type,Engine,IMPRESSIONS,SPEND,CLICKS,CONVERSIONS
0,2025-01,AMICA AUTO - BRAND,Brand,GOOGLE,14789,145271.48,5629,1584.51
1,2025-01,AMICA AUTO - BRAND - BING,Brand,BING,2887,24502.47,869,313.00
2,2025-01,AMICA BRAND - CONDO INSURANCE,Brand,GOOGLE,83,1025.64,33,8.04
3,2025-01,AMICA BRAND - ENTERPRISE - NON STRATEGIC - BING,Brand,BING,5995,15407.74,1694,360.00
4,2025-01,AMICA BRAND - ENTERPRISE - NON-STRATEGIC,Brand,GOOGLE,13130,42192.42,5008,1478.36


In [504]:
exact_merged = pd.merge(engine, internal_grouped, left_on=['lead_monyr', 'CAMPAIGN_NAME', 'keyword_type', 'Engine'],right_on=['lead_monyr', 'campaign', 'keyword_type', 'Engine'], how='inner')

In [505]:

exact_merged_test = pd.merge(
    engine,
    internal_grouped,
    left_on=['lead_monyr', 'CAMPAIGN_NAME', 'keyword_type', 'Engine'],
    right_on=['lead_monyr', 'campaign', 'keyword_type', 'Engine'],
    how='outer',  # use 'outer' to capture all unmatched rows
    indicator=True
)


In [506]:
engine_unmatched = exact_merged_test[exact_merged_test['_merge'] == 'left_only']

In [507]:
internal_unmatched = exact_merged_test[exact_merged_test['_merge'] == 'right_only']

In [508]:
engine_unmatched.CONVERSIONS.sum()

128.24

In [509]:
internal_unmatched.lead.sum()

1593.0

In [510]:
exact_merged = exact_merged.drop(columns=['campaign'])
engine_unmatched = engine_unmatched.drop(columns=['_merge'])
internal_unmatched = internal_unmatched.drop(columns=['_merge'])
internal_unmatched = internal_unmatched.drop(columns=['CAMPAIGN_NAME'])

In [511]:
internal_unmatched.rename(columns={
    'campaign': 'CAMPAIGN_NAME'
}, inplace=True)

In [512]:
internal_unmatched.head()

,lead_monyr,keyword_type,Engine,IMPRESSIONS,SPEND,CLICKS,CONVERSIONS,CAMPAIGN_NAME,lead,quote,issue,day0,day7,Auto_Phone_leads,Auto_Web_leads,Seg_1_2_Auto_Web_leads,Seg_0_Auto_Web_leads,Seg_1_2_Auto_Phone_leads,Seg_0_Auto_Phone_leads,Auto_Issues,Home_Phone_leads,Home_Web_leads,Home_Issues,Auto_Day7,Home_Day7,Auto_Web_Issues,Auto_Phone_Issues,Auto_Web_Day7,Home_Web_Issues,Home_Phone_Issues,Home_Web_Day7
25,2025-01,PMAX,GOOGLE,NaN,NaN,NaN,NaN,AUTO GENERIC - ME - PERFORMANCE MAX,112.0,112.0,0.0,0.0,0.0,0,99.0,27.0,0.0,0.0,0.0,0.0,0,13.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0
45,2025-01,Generic,GOOGLE,NaN,NaN,NaN,NaN,CONDO - GENERIC - STRATEGIC,24.0,24.0,0.0,0.0,0.0,0,3.0,1.0,0.0,0.0,0.0,0.0,0,21.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0
48,2025-01,Brand,BING,NaN,NaN,NaN,NaN,ENTERPRISE BRAND-ALL-CALL MARKETS,2.0,2.0,0.0,0.0,0.0,0,1.0,1.0,0.0,0.0,0.0,0.0,0,1.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0
84,2025-01,Brand,GOOGLE,NaN,NaN,NaN,NaN,LIFE - BRAND,16.0,16.0,0.0,0.0,0.0,0,9.0,7.0,0.0,0.0,0.0,0.0,0,7.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0
85,2025-01,Generic,GOOGLE,NaN,NaN,NaN,NaN,LIFE GENERIC - GA,2.0,2.0,0.0,0.0,0.0,0,1.0,0.0,0.0,0.0,0.0,0.0,0,1.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0


In [513]:
internal_unmatched = internal_unmatched[['lead_monyr', 'CAMPAIGN_NAME', 'keyword_type', 'Engine', 'IMPRESSIONS',
       'SPEND', 'CLICKS', 'CONVERSIONS', 'lead', 'quote', 'issue', 'day0',
       'day7', 'Auto_Phone_leads', 'Auto_Web_leads', 'Seg_1_2_Auto_Web_leads',
       'Seg_0_Auto_Web_leads', 'Seg_1_2_Auto_Phone_leads',
       'Seg_0_Auto_Phone_leads', 'Auto_Issues', 'Home_Phone_leads',
       'Home_Web_leads', 'Home_Issues', 'Auto_Day7', 'Home_Day7',
       'Auto_Web_Issues', 'Auto_Phone_Issues', 'Auto_Web_Day7',
       'Home_Web_Issues', 'Home_Phone_Issues', 'Home_Web_Day7']]

In [514]:
engine_unmatched = engine_unmatched.drop(columns=['campaign'])

In [515]:
Search_Web = pd.concat([exact_merged, engine_unmatched, internal_unmatched], ignore_index=True)

In [516]:
Search_Web = Search_Web.fillna(0)

/tmp/ipykernel_1887463/84554327.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Search_Web = Search_Web.fillna(0)


In [517]:
Search_Web.head()

,lead_monyr,CAMPAIGN_NAME,keyword_type,Engine,IMPRESSIONS,SPEND,CLICKS,CONVERSIONS,lead,quote,issue,day0,day7,Auto_Phone_leads,Auto_Web_leads,Seg_1_2_Auto_Web_leads,Seg_0_Auto_Web_leads,Seg_1_2_Auto_Phone_leads,Seg_0_Auto_Phone_leads,Auto_Issues,Home_Phone_leads,Home_Web_leads,Home_Issues,Auto_Day7,Home_Day7,Auto_Web_Issues,Auto_Phone_Issues,Auto_Web_Day7,Home_Web_Issues,Home_Phone_Issues,Home_Web_Day7
0,2025-01,AMICA AUTO - BRAND,Brand,GOOGLE,14789.0,145271.48,5629.0,1584.51,2004.0,2004.0,59.0,12.0,33.0,0,1518.0,728.0,3.0,0.0,0.0,46.0,0,486.0,13.0,27.0,6.0,46.0,0,27.0,13.0,0,0
1,2025-01,AMICA AUTO - BRAND - BING,Brand,BING,2887.0,24502.47,869.0,313.00,370.0,370.0,24.0,2.0,8.0,0,278.0,149.0,2.0,0.0,0.0,18.0,0,92.0,6.0,7.0,1.0,18.0,0,7.0,6.0,0,0
2,2025-01,AMICA BRAND - CONDO INSURANCE,Brand,GOOGLE,83.0,1025.64,33.0,8.04,11.0,11.0,0.0,0.0,0.0,0,3.0,3.0,0.0,0.0,0.0,0.0,0,8.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0
3,2025-01,AMICA BRAND - ENTERPRISE - NON STRATEGIC - BING,Brand,BING,5995.0,15407.74,1694.0,360.00,532.0,532.0,8.0,0.0,3.0,0,311.0,223.0,0.0,0.0,0.0,4.0,0,221.0,4.0,1.0,2.0,4.0,0,1.0,4.0,0,0
4,2025-01,AMICA BRAND - ENTERPRISE - NON-STRATEGIC,Brand,GOOGLE,13130.0,42192.42,5008.0,1478.36,1614.0,1614.0,26.0,2.0,5.0,0,903.0,641.0,2.0,0.0,0.0,16.0,0,711.0,10.0,5.0,0.0,16.0,0,5.0,10.0,0,0


In [518]:
Search_Web.lead.sum()

272405.0

In [519]:
#Search_Web.to_csv('/data/Users/Shelly/Search Report/Search_Web.csv')

### Phone Leads

In [520]:
query2 = f'''

select a.lead_storage_nbr, a.line_of_business,a.new_segment_reporting_cd as quality_seg,
a.lead_dt, a.lead_contact_method,
a.lead_medium,b.medium_channel,b.medium,b.source,b.campaign,
sum(a.lead)lead,sum(quote)quote,sum(a.issued_policy)issue,
sum(case when lead_dt=current_proc_dt then issued_policy else 0 end ) Day0,
sum(case when current_proc_dt - lead_dt <=6 then issued_policy else 0 end ) Day7,
state_cd,
case when state_cd in('CT','ME','NY','PA','RI','VT','NH','MA') then 'Target'
     when state_cd in('AZ','DC','DE','MD','NC','NM','OR','VA','WA') then 'Tier2'
else 'Non-Target' 
end state_grp,
to_char(lead_dt,'YYYY-MM') lead_monyr
 
from  PROD_DB_PII_ANALYTICS.SANDBOX.leads_policies_auto_home_pup  a,
PROD_DB_PII_ANALYTICS.SANDBOX.last_touch_attribution_vq b
where a.lead_Dt between'{start_date}' AND '{end_date}'
and b.medium in('cpc','Paid Search')
and a.lead_storage_nbr=b.orig_storage_no
group by 1,2,3,4,5,6,7,8,9,10,16,17,18;

'''

internal_data = pd.read_sql(query2, ctx)
print(internal_data.shape)
internal_data.head()

/tmp/ipykernel_1887463/1854016874.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  internal_data = pd.read_sql(query2, ctx)


(325408, 18)


,LEAD_STORAGE_NBR,LINE_OF_BUSINESS,QUALITY_SEG,LEAD_DT,LEAD_CONTACT_METHOD,LEAD_MEDIUM,MEDIUM_CHANNEL,MEDIUM,SOURCE,CAMPAIGN,LEAD,QUOTE,ISSUE,DAY0,DAY7,STATE_CD,STATE_GRP,LEAD_MONYR
0,Q007271432,AUTOMOBILE,0,2025-02-25,PHONE,Paid Search,Paid Search,Paid Search,None,Home Call Extension - Conversion Tracking,1,0,0,0,0,CT,Target,2025-02
1,W012406931,AUTOMOBILE,4,2025-11-03,WEB,Paid Search,Paid Search,cpc,google,auto generic wa performance max,1,1,0,0,0,WA,Tier2,2025-11
2,W012331841,AUTOMOBILE,2,2025-10-14,WEB,Paid Search,Paid Search,cpc,google,auto generic nh performance max,1,1,0,0,0,VA,Tier2,2025-10
3,W011871040,AUTOMOBILE,1,2025-06-23,WEB,Paid Search,Paid Search,cpc,google,enterprise - non - strategic,1,1,0,0,0,OH,Non-Target,2025-06
4,W012196280,AUTOMOBILE,1,2025-09-11,WEB,Paid Search,Paid Search,cpc,google,home generic ri performance max,1,1,0,0,0,MA,Target,2025-09


In [521]:
#internal_data = pd.read_csv('/data/Projects/lead_policy_counts/aggregated_df_search.csv')

In [522]:
internal_data.columns = internal_data.columns.str.lower()

In [523]:
internal_data = internal_data[internal_data['lead_contact_method'] == 'PHONE']

In [524]:
internal_data.shape

(44632, 18)

In [525]:
exclude_list = [
    'QS_AUTO_INSURANCE', 'MA_AUTO_INSURANCE', 'MA_HOME_INSURANCE', 'CA_AUTO_INSURANCE',
    'USNEWS_HOME_INSURANCE', 'CNBC', 'CA_HOME_INSURANCE', 'MONEYGROUP_TIER1PUBS',
    'BEST_MONEY_AUTO_INSURANCE', 'MONEY.COM', '2501R'
]

internal_data = internal_data[
    ~internal_data['campaign'].str.upper().isin(exclude_list)
]

In [526]:

import pandas as pd

def update_campaign_name(df: pd.DataFrame) -> pd.DataFrame:
    """
    Update df['campaign'] based on specific rules involving 'campaign' and 'state_cd'.
    Rules are applied in sequence. If a rule does not match, the campaign stays as-is.

    Required columns: 'campaign', 'state_cd'
    """

    # --- Validate required columns ---
    required = {'campaign', 'state_cd'}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    # Work on a copy (order of assignments controls precedence)
    out = df['campaign'].copy()

    # --- Shared state sets ---
    states_ct_az = {'CT', 'AZ'}
    strategic_states = {'AZ', 'CT', 'DC', 'DE', 'MA', 'MD', 'ME', 'NH', 'NM', 'NY', 'OR', 'PA', 'RI', 'VT', 'WA', 'VA'}

    # --- Campaign groups from earlier rules ---
    auto_pdps = {
        'Amica.com Product Development Paid Search - Google (Auto)',
        'Amica.com Product Development Paid Search -Bing (Auto)',  # exact per your spec
    }
    home_pdps = {
        'Amica.com Product Development Paid Search - Google (Home)',
        'Amica.com Product Development Paid Search - Bing (Home)',
    }
    enterprise_campaigns = {
        'Enterprise - Conversion Tracking',
        'Brand - Google - Desktop',
        'Enterprise Paid Search Site-desktop only',
        'Enterprise Call Extension - Conversion Tracking',
        'Paid Search Bundle Message Test - Convenience message',
    }
    auto_conv = {
        'Google Auto Generic - Conversion Tracking',
        'Google Auto Call Extension - Conversion Tracking',
        'Bing Auto - Conversion Tracking',
        'Bing Auto Call Extension - Conversion Tracking',
    }

    # --- New campaign names (previous additions) ---
    auto_bundle_google = 'Auto Bundle Paid Search Campaign - Google'
    auto_bundle_bing   = 'Auto Bundle Paid Search Campaign - Bing'
    pmax_home_campaign = 'PMAX Home Campaign - Test States NH, CT, RI'
    bing_enterprise_campaigns = {
        'Bing Enterprise Call Extension - Conversion Tracking',
        'Bing Enterprise - Conversion Tracking',
    }

    # ============================================================
    # 1) PDPs (Auto): CT/AZ -> AUTO GENERIC - {state}
    # ============================================================
    m1 = df['campaign'].isin(auto_pdps) & df['state_cd'].isin(states_ct_az)
    out.loc[m1] = 'AUTO GENERIC - ' + df.loc[m1, 'state_cd']

    # ============================================================
    # 2) PDPs (Home): CT/AZ -> HOME GENERIC - {state}
    # ============================================================
    m2 = df['campaign'].isin(home_pdps) & df['state_cd'].isin(states_ct_az)
    out.loc[m2] = 'HOME GENERIC - ' + df.loc[m2, 'state_cd']

    # ============================================================
    # 3) Enterprise (non-Bing)
    # ============================================================
    m3 = df['campaign'].isin(enterprise_campaigns)
    m3_strat = m3 & df['state_cd'].isin(strategic_states)
    m3_non   = m3 & ~df['state_cd'].isin(strategic_states)
    out.loc[m3_strat] = 'AMICA BRAND - ENTERPRISE - STRATEGIC'
    out.loc[m3_non]   = 'AMICA BRAND - ENTERPRISE - NON-STRATEGIC'

    # ============================================================
    # 4) Auto conversion tracking -> AUTO GENERIC - {state}
    # ============================================================
    m4 = df['campaign'].isin(auto_conv) & df['state_cd'].isin(strategic_states)
    out.loc[m4] = 'AUTO GENERIC - ' + df.loc[m4, 'state_cd']

    # ============================================================
    # 5) Home conversion tracking (split into non-Bing and Bing)
    #    Non-Bing -> HOME GENERIC - {state}
    #    Bing     -> HOME GENERIC - {state} - BING
    # ============================================================
    home_conv_non_bing = {
        'Home - Conversion Tracking',
        'Home Generic',
        'Home Call Extension - Conversion Tracking',
    }
    home_conv_bing = {
        'Bing Home - Conversion Tracking',
        'Bing Home Call Extension - Conversion Tracking',
    }
    m5a = df['campaign'].isin(home_conv_non_bing) & df['state_cd'].isin(strategic_states)
    out.loc[m5a] = 'HOME GENERIC - ' + df.loc[m5a, 'state_cd']
    m5b = df['campaign'].isin(home_conv_bing) & df['state_cd'].isin(strategic_states)
    out.loc[m5b] = 'HOME GENERIC - ' + df.loc[m5b, 'state_cd'] + ' - BING'

    # ============================================================
    # 6) Auto Bundle (Google): PA/NH/RI/CT -> Auto Generic - {state} - Bundle
    # ============================================================
    m6 = (df['campaign'] == auto_bundle_google) & df['state_cd'].isin({'PA', 'NH', 'RI', 'CT'})
    out.loc[m6] = 'Auto Generic - ' + df.loc[m6, 'state_cd'] + ' - Bundle'

    # ============================================================
    # 7) Auto Bundle (Bing):
    #    PA/CT -> Auto Generic - {state} - Bundle
    #    NH/RI -> Auto Generic - NH | RI - Bundle
    # ============================================================
    m7a = (df['campaign'] == auto_bundle_bing) & df['state_cd'].isin({'PA', 'CT'})
    out.loc[m7a] = 'Auto Generic - ' + df.loc[m7a, 'state_cd'] + ' - Bundle'
    m7b = (df['campaign'] == auto_bundle_bing) & df['state_cd'].isin({'NH', 'RI'})
    out.loc[m7b] = 'Auto Generic - NH | RI - Bundle'

    # ============================================================
    # 8) PMAX Home campaign:
    #    NH/MA/VT -> Home Generic - {state}- Performance Max (no space before hyphen)
    # ============================================================
    m8 = (df['campaign'] == pmax_home_campaign) & df['state_cd'].isin({'NH', 'MA', 'VT','CT','RI'})
    out.loc[m8] = 'Home Generic - ' + df.loc[m8, 'state_cd'] + ' - Performance Max'

    # ============================================================
    # 9) Bing Enterprise group:
    #    strategic -> Amica Brand - Enteprise - Strategic - Bing
    #    else      -> Amica Brand - Enteprise - Non Strategic - Bing
    # ============================================================
    m9 = df['campaign'].isin(bing_enterprise_campaigns)
    m9_strat = m9 & df['state_cd'].isin(strategic_states)
    m9_non   = m9 & ~df['state_cd'].isin(strategic_states)
    out.loc[m9_strat] = 'AMICA BRAND - ENTERPRISE - STRATEGIC - BING'
    out.loc[m9_non]   = 'AMICA BRAND - ENTERPRISE - NON STRATEGIC - BING'

    # ============================================================
    # 10) NEW: Brand renames (Google/Bing)
    # ============================================================
    m10a = (df['campaign'] == 'Amica Auto Brand campaign (Google)')
    out.loc[m10a] = 'Amica Auto - Brand'
    m10b = (df['campaign'] == 'Amica Home Brand campaign (Google)')
    out.loc[m10b] = 'Amica Brand - Home Insurance'
    m10c = (df['campaign'] == 'Amica Auto Brand Campaign (Bing)')
    out.loc[m10c] = 'Amica Auto - Brand - Bing'
    m10d = (df['campaign'] == 'Amica Home Brand Campaign (Bing)')
    out.loc[m10d] = 'Amica Brand - Home Insurance - Bing'

    # ============================================================
    # 11) NEW: PMAX states logic for two campaign names
    # ============================================================
    pmax_states = {'CT', 'MA', 'WA', 'RI', 'NY', 'NH', 'ME', 'VT', 'PA'}
    pmax_campaigns = {'AUTO GENERIC - PERFORMANCE MAX', 'Pmax Test Pilot'}
    m11 = df['campaign'].isin(pmax_campaigns) & df['state_cd'].isin(pmax_states)
    out.loc[m11] = 'AUTO GENERIC - ' + df.loc[m11, 'state_cd'] + ' - PERFORMANCE MAX'

    # ============================================================
    # 12) NEW: Condo Call Extension - Conversion Tracking -> Condo Insurance - Generic
    # ============================================================
    m12 = (df['campaign'] == 'Condo Call Extension - Conversion Tracking')
    out.loc[m12] = 'Condo Insurance - Generic'

    # --- Write back and return ---
    df = df.copy()
    df['campaign'] = out
    return df


def update_campaign_name(df):
    state_list = {'AZ', 'CT', 'DC', 'DE', 'MA', 'MD', 'ME', 'NH', 'NM', 'NY', 'OR', 'PA', 'RI', 'VT', 'WA', 'VA'}

    def generate_name(row):
        state = row['state_cd']
        lob = row['line_of_business']
        
        if state in state_list:
            if lob == 'HOMEOWNERS':
                return f'HOME GENERIC - {state}'
            elif lob == 'AUTOMOBILE':
                return f'AUTO GENERIC - {state}'
        else:
            if lob == 'HOMEOWNERS':
                return 'AMICA BRAND - HOME INSURANCE'
            elif lob == 'AUTOMOBILE':
                return 'AMICA AUTO - BRAND'
        return row['campaign']  # fallback to original if none match

    df['campaign'] = df.apply(generate_name, axis=1)
    return df


In [527]:
internal_data = update_campaign_name(internal_data)

In [528]:
internal_data.head()

,lead_storage_nbr,line_of_business,quality_seg,lead_dt,lead_contact_method,lead_medium,medium_channel,medium,source,campaign,lead,quote,issue,day0,day7,state_cd,state_grp,lead_monyr
0,Q007271432,AUTOMOBILE,0,2025-02-25,PHONE,Paid Search,Paid Search,Paid Search,None,HOME GENERIC - CT,1,0,0,0,0,CT,Target,2025-02
18,Q008643678,AUTOMOBILE,2,2025-10-27,PHONE,Paid Search,Paid Search,Paid Search,None,Amica Auto - Brand,1,1,0,0,0,VA,Tier2,2025-10
28,Q008491001,AUTOMOBILE,0,2025-09-28,PHONE,Paid Search,Paid Search,Paid Search,None,Amica Auto - Brand,1,0,0,0,0,NY,Target,2025-09
37,Q007087517,AUTOMOBILE,4,2025-01-24,PHONE,Paid Search,Paid Search,Paid Search,None,AMICA BRAND - ENTERPRISE - NON-STRATEGIC,1,1,0,0,0,WI,Non-Target,2025-01
59,Q007995649,AUTOMOBILE,3,2025-07-02,PHONE,Paid Search,Paid Search,Paid Search,None,Auto Bundle Paid Search Campaign - Bing,1,1,1,1,1,MA,Target,2025-07


In [529]:
internal_data['lead_dt'].max()

datetime.date(2026, 1, 24)

In [530]:
start_date1 = pd.to_datetime(start_date1)
end_date1 = pd.to_datetime(end_date1)
internal_data['lead_dt'] = pd.to_datetime(internal_data['lead_dt'])
internal_data = internal_data[(internal_data['lead_dt'] >= start_date1) &(internal_data['lead_dt'] <= end_date1)]
internal_data = internal_data[internal_data['medium'].isin(['Paid Search', 'cpc'])]

In [531]:
internal_data['lead_dt'] = pd.to_datetime(internal_data['lead_dt'])
internal_data['lead_monyr'] = internal_data['lead_dt'].dt.strftime('%Y-%m')

In [532]:
internal_data['campaign'] = internal_data['campaign'].astype(str).str.upper()

In [533]:
def classify_keyword(campaign):
    campaign_upper = str(campaign).upper()
    if 'BRAND' in campaign_upper or 'ENTERPRISE' in campaign_upper:
        return 'Brand'
    elif 'MAX' in campaign_upper:
        return 'PMAX'
    else:
        return 'Generic'

internal_data['keyword_type'] = internal_data['campaign'].apply(classify_keyword)

In [534]:
internal_data['Auto_Phone_leads'] = internal_data.apply(lambda row: row['lead'] if row['line_of_business'] == 'AUTOMOBILE' and row['lead_contact_method'] == 'PHONE' else None,axis=1)
internal_data['Auto_Web_leads'] = internal_data.apply(lambda row: row['lead'] if row['line_of_business'] == 'AUTOMOBILE' and row['lead_contact_method'] == 'WEB' else None,axis=1)
internal_data['Seg_1_2_Auto_Web_leads'] = internal_data.apply(lambda row: row['Auto_Web_leads'] if row['quality_seg'] in [1, 2] else 0,axis=1)
internal_data['Seg_0_Auto_Web_leads'] = internal_data.apply(lambda row: 1 if (row['line_of_business'] == 'AUTOMOBILE' and row['quality_seg'] == 0 and row['lead_contact_method'] == 'WEB') else 0,axis=1)
internal_data['Seg_1_2_Auto_Phone_leads'] = internal_data.apply(lambda row: row['Auto_Phone_leads'] if row['quality_seg'] in [1, 2] else 0,axis=1)
internal_data['Seg_0_Auto_Phone_leads'] = internal_data.apply(lambda row: 1 if (row['line_of_business'] == 'AUTOMOBILE' and row['quality_seg'] == 0 and row['lead_contact_method'] == 'PHONE') else 0,axis=1)
internal_data['Auto_Issues'] = internal_data.apply(lambda row: row['issue'] if row['line_of_business'] == 'AUTOMOBILE' else None,axis=1)
internal_data['Auto_Web_Issues'] = internal_data.apply(lambda row: row['issue'] if (row['line_of_business'] == 'AUTOMOBILE' and row['lead_contact_method'] == 'WEB') else None, axis=1)
internal_data['Auto_Phone_Issues'] = internal_data.apply(lambda row: row['issue'] if (row['line_of_business'] == 'AUTOMOBILE' and row['lead_contact_method'] == 'PHONE') else None, axis=1)
internal_data['Auto_Day7'] = internal_data.apply(lambda row: row['day7'] if row['line_of_business'] == 'AUTOMOBILE' else None,axis=1)
internal_data['Auto_Web_Day7'] = internal_data.apply(lambda row: row['day7'] if (row['line_of_business'] == 'AUTOMOBILE' and row['lead_contact_method'] == 'WEB') else None,axis=1)
internal_data['Auto_Phone_Day7'] = internal_data.apply(lambda row: row['day7'] if (row['line_of_business'] == 'AUTOMOBILE' and row['lead_contact_method'] == 'PHONE') else None,axis=1)

In [535]:
internal_data['Home_Phone_leads'] = internal_data.apply(lambda row: row['lead'] if row['line_of_business'] == 'HOMEOWNERS' and row['lead_contact_method'] == 'PHONE' else None,axis=1)
internal_data['Home_Web_leads'] = internal_data.apply(lambda row: row['lead'] if row['line_of_business'] == 'HOMEOWNERS' and row['lead_contact_method'] == 'WEB' else None,axis=1)
internal_data['Home_Issues'] = internal_data.apply(lambda row: row['issue'] if row['line_of_business'] == 'HOMEOWNERS' else None,axis=1)
internal_data['Home_Web_Issues'] = internal_data.apply(lambda row: row['issue'] if (row['line_of_business'] == 'HOMEOWNERS' and row['lead_contact_method'] == 'WEB') else None,axis=1)
internal_data['Home_Phone_Issues'] = internal_data.apply(lambda row: row['issue'] if (row['line_of_business'] == 'HOMEOWNERS' and row['lead_contact_method'] == 'PHONE') else None,axis=1)
internal_data['Home_Day7'] = internal_data.apply(lambda row: row['day7'] if row['line_of_business'] == 'HOMEOWNERS' else None,axis=1)
internal_data['Home_Web_Day7'] = internal_data.apply(lambda row: row['day7'] if (row['line_of_business'] == 'HOMEOWNERS' and row['lead_contact_method'] == 'WEB') else None,axis=1)
internal_data['Home_Web_Day7'] = internal_data.apply(lambda row: row['day7'] if (row['line_of_business'] == 'HOMEOWNERS' and row['lead_contact_method'] == 'PHONE') else None,axis=1)

In [536]:
# Assume `df` is your main dataset and `lookup_df` is your lookup table

# Create a dictionary from the lookup table
lookup_dict = dict(zip(lookup_df['int_names'], lookup_df['eng_names']))

# Replace campaign names using the dictionary
internal_data['campaign'] = internal_data['campaign'].apply(lambda x: lookup_dict.get(x, x))

In [537]:

internal_data['Engine'] = np.where(
    internal_data['campaign'].str.contains('bing', case=False, na=False),
    'Bing',
    'Google'
)


In [538]:
# Group Google Ads data
google_grouped = google_data.groupby(['lead_monyr', 'CAMPAIGN_NAME','keyword_type','Engine'], as_index=False).agg({
    'IMPRESSIONS': 'sum',
    'SPEND': 'sum',
    'CLICKS': 'sum',
    'CONVERSIONS': 'sum'
})

# Group Google Ads data
bing_grouped = bing_data.groupby(['lead_monyr', 'CAMPAIGN_NAME','keyword_type','Engine'], as_index=False).agg({
    'IMPRESSIONS': 'sum',
    'SPEND': 'sum',
    'CLICKS': 'sum',
    'CONVERSIONS': 'sum'
})

# Group Internal Company data
internal_grouped = internal_data.groupby(['lead_monyr', 'campaign','keyword_type','Engine'], as_index=False).agg({
     'lead': 'sum',
    'quote': 'sum',
    'issue': 'sum',
    'day0': 'sum',
    'day7': 'sum',
    'Auto_Phone_leads' :'sum',
    'Auto_Web_leads':'sum',
    'Seg_1_2_Auto_Web_leads':'sum',
    'Seg_0_Auto_Web_leads':'sum',
    'Seg_1_2_Auto_Phone_leads':'sum',
    'Seg_0_Auto_Phone_leads':'sum',
    'Auto_Issues':'sum',
    'Home_Phone_leads':'sum',
    'Home_Web_leads':'sum',
    'Home_Issues':'sum',
    'Auto_Day7':'sum',
    'Home_Day7':'sum',
    'Auto_Web_Issues' :'sum',
    'Auto_Phone_Issues' :'sum',
    'Auto_Web_Day7' :'sum',
    'Auto_Web_Day7' :'sum',
    'Home_Web_Issues' :'sum',
    'Home_Phone_Issues' :'sum',
    'Home_Web_Day7' :'sum',
    'Home_Web_Day7' :'sum'
})

In [539]:
# Convert campaign names to uppercase
engine['CAMPAIGN_NAME'] = engine['CAMPAIGN_NAME'].str.upper()
engine['Engine'] = engine['Engine'].str.upper()
internal_grouped['campaign'] = internal_grouped['campaign'].str.upper()
internal_grouped['Engine'] = internal_grouped['Engine'].str.upper()

In [540]:
engine.head()

,lead_monyr,CAMPAIGN_NAME,keyword_type,Engine,IMPRESSIONS,SPEND,CLICKS,CONVERSIONS
0,2025-01,AMICA AUTO - BRAND,Brand,GOOGLE,14789,145271.48,5629,1584.51
1,2025-01,AMICA AUTO - BRAND - BING,Brand,BING,2887,24502.47,869,313.00
2,2025-01,AMICA BRAND - CONDO INSURANCE,Brand,GOOGLE,83,1025.64,33,8.04
3,2025-01,AMICA BRAND - ENTERPRISE - NON STRATEGIC - BING,Brand,BING,5995,15407.74,1694,360.00
4,2025-01,AMICA BRAND - ENTERPRISE - NON-STRATEGIC,Brand,GOOGLE,13130,42192.42,5008,1478.36


In [541]:
internal_grouped.head()

,lead_monyr,campaign,keyword_type,Engine,lead,quote,issue,day0,day7,Auto_Phone_leads,Auto_Web_leads,Seg_1_2_Auto_Web_leads,Seg_0_Auto_Web_leads,Seg_1_2_Auto_Phone_leads,Seg_0_Auto_Phone_leads,Auto_Issues,Home_Phone_leads,Home_Web_leads,Home_Issues,Auto_Day7,Home_Day7,Auto_Web_Issues,Auto_Phone_Issues,Auto_Web_Day7,Home_Web_Issues,Home_Phone_Issues,Home_Web_Day7
0,2025-01,AMICA AUTO - BRAND,Brand,GOOGLE,333,285,57,21,40,265.0,0,0.0,0,111.0,39,40.0,68.0,0,17.0,27.0,13.0,0,40.0,0,0,17.0,13.0
1,2025-01,AMICA AUTO - BRAND - BING,Brand,BING,25,21,7,2,3,19.0,0,0.0,0,9.0,2,4.0,6.0,0,3.0,2.0,1.0,0,4.0,0,0,3.0,1.0
2,2025-01,AMICA BRAND - ENTERPRISE - NON STRATEGIC - BING,Brand,BING,93,84,6,3,6,52.0,0,0.0,0,33.0,1,2.0,41.0,0,4.0,2.0,4.0,0,2.0,0,0,4.0,4.0
3,2025-01,AMICA BRAND - ENTERPRISE - NON-STRATEGIC,Brand,GOOGLE,331,268,24,4,10,183.0,0,0.0,0,95.0,26,13.0,148.0,0,11.0,6.0,4.0,0,13.0,0,0,11.0,4.0
4,2025-01,AMICA BRAND - ENTERPRISE - STRATEGIC,Brand,GOOGLE,796,677,113,36,72,535.0,0,0.0,0,222.0,84,54.0,261.0,0,59.0,37.0,35.0,0,54.0,0,0,59.0,35.0


In [542]:
exact_merged = pd.merge(engine, internal_grouped, left_on=['lead_monyr', 'CAMPAIGN_NAME', 'keyword_type', 'Engine'],right_on=['lead_monyr', 'campaign', 'keyword_type', 'Engine'], how='inner')

In [543]:

exact_merged_test = pd.merge(
    engine,
    internal_grouped,
    left_on=['lead_monyr', 'CAMPAIGN_NAME', 'keyword_type', 'Engine'],
    right_on=['lead_monyr', 'campaign', 'keyword_type', 'Engine'],
    how='outer',  # use 'outer' to capture all unmatched rows
    indicator=True
)


In [544]:
engine_unmatched = exact_merged_test[exact_merged_test['_merge'] == 'left_only']
internal_unmatched = exact_merged_test[exact_merged_test['_merge'] == 'right_only']

In [545]:
engine_unmatched.CONVERSIONS.sum()

33029.17

In [546]:
internal_unmatched['lead'].sum()

5463.0

In [547]:
exact_merged = exact_merged.drop(columns=['campaign'])
engine_unmatched = engine_unmatched.drop(columns=['_merge'])
internal_unmatched = internal_unmatched.drop(columns=['_merge'])
internal_unmatched = internal_unmatched.drop(columns=['CAMPAIGN_NAME'])

In [548]:
exact_merged.head(3)

,lead_monyr,CAMPAIGN_NAME,keyword_type,Engine,IMPRESSIONS,SPEND,CLICKS,CONVERSIONS,lead,quote,issue,day0,day7,Auto_Phone_leads,Auto_Web_leads,Seg_1_2_Auto_Web_leads,Seg_0_Auto_Web_leads,Seg_1_2_Auto_Phone_leads,Seg_0_Auto_Phone_leads,Auto_Issues,Home_Phone_leads,Home_Web_leads,Home_Issues,Auto_Day7,Home_Day7,Auto_Web_Issues,Auto_Phone_Issues,Auto_Web_Day7,Home_Web_Issues,Home_Phone_Issues,Home_Web_Day7
0,2025-01,AMICA AUTO - BRAND,Brand,GOOGLE,14789,145271.48,5629,1584.51,333,285,57,21,40,265.0,0,0.0,0,111.0,39,40.0,68.0,0,17.0,27.0,13.0,0,40.0,0,0,17.0,13.0
1,2025-01,AMICA AUTO - BRAND - BING,Brand,BING,2887,24502.47,869,313.00,25,21,7,2,3,19.0,0,0.0,0,9.0,2,4.0,6.0,0,3.0,2.0,1.0,0,4.0,0,0,3.0,1.0
2,2025-01,AMICA BRAND - ENTERPRISE - NON STRATEGIC - BING,Brand,BING,5995,15407.74,1694,360.00,93,84,6,3,6,52.0,0,0.0,0,33.0,1,2.0,41.0,0,4.0,2.0,4.0,0,2.0,0,0,4.0,4.0


In [549]:
engine_unmatched.head(3)

,lead_monyr,CAMPAIGN_NAME,keyword_type,Engine,IMPRESSIONS,SPEND,CLICKS,CONVERSIONS,campaign,lead,quote,issue,day0,day7,Auto_Phone_leads,Auto_Web_leads,Seg_1_2_Auto_Web_leads,Seg_0_Auto_Web_leads,Seg_1_2_Auto_Phone_leads,Seg_0_Auto_Phone_leads,Auto_Issues,Home_Phone_leads,Home_Web_leads,Home_Issues,Auto_Day7,Home_Day7,Auto_Web_Issues,Auto_Phone_Issues,Auto_Web_Day7,Home_Web_Issues,Home_Phone_Issues,Home_Web_Day7
2,2025-01,AMICA BRAND - CONDO INSURANCE,Brand,GOOGLE,83.0,1025.64,33.0,8.04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,2025-01,AUTO GENERIC - AZ,Generic,BING,69.0,889.15,9.0,1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15,2025-01,AUTO GENERIC - CT,Generic,BING,8162.0,22930.57,459.0,136.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [550]:
internal_unmatched.head(3)

,lead_monyr,keyword_type,Engine,IMPRESSIONS,SPEND,CLICKS,CONVERSIONS,campaign,lead,quote,issue,day0,day7,Auto_Phone_leads,Auto_Web_leads,Seg_1_2_Auto_Web_leads,Seg_0_Auto_Web_leads,Seg_1_2_Auto_Phone_leads,Seg_0_Auto_Phone_leads,Auto_Issues,Home_Phone_leads,Home_Web_leads,Home_Issues,Auto_Day7,Home_Day7,Auto_Web_Issues,Auto_Phone_Issues,Auto_Web_Day7,Home_Web_Issues,Home_Phone_Issues,Home_Web_Day7
9,2025-01,Generic,GOOGLE,NaN,NaN,NaN,NaN,AMICA.COM PRODUCT DEVELOPMENT PAID SEARCH - GO...,1.0,1.0,0.0,0.0,0.0,1.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0,0.0,0,0,0.0,0.0
10,2025-01,Generic,GOOGLE,NaN,NaN,NaN,NaN,AMICA.COM PRODUCT DEVELOPMENT PAID SEARCH - GO...,17.0,14.0,2.0,0.0,0.0,3.0,0,0.0,0.0,2.0,1.0,1.0,14.0,0,1.0,0.0,0.0,0,1.0,0,0,1.0,0.0
11,2025-01,Generic,BING,NaN,NaN,NaN,NaN,AUTO BUNDLE PAID SEARCH CAMPAIGN - BING,34.0,29.0,5.0,1.0,1.0,20.0,0,0.0,0.0,12.0,2.0,2.0,14.0,0,3.0,0.0,1.0,0,2.0,0,0,3.0,1.0


In [551]:
internal_unmatched.rename(columns={
    'campaign': 'CAMPAIGN_NAME'
}, inplace=True)

In [552]:
internal_unmatched = internal_unmatched[['lead_monyr','CAMPAIGN_NAME', 'keyword_type', 'Engine', 'IMPRESSIONS', 'SPEND',
       'CLICKS', 'CONVERSIONS', 'lead', 'quote', 'issue',
       'day0', 'day7', 'Auto_Phone_leads', 'Auto_Web_leads',
       'Seg_1_2_Auto_Web_leads', 'Seg_0_Auto_Web_leads',
       'Seg_1_2_Auto_Phone_leads', 'Seg_0_Auto_Phone_leads', 'Auto_Issues',
       'Home_Phone_leads', 'Home_Web_leads', 'Home_Issues', 'Auto_Day7',
       'Home_Day7',
'Auto_Web_Issues', 'Auto_Phone_Issues', 'Auto_Web_Day7',
    'Home_Web_Issues', 'Home_Phone_Issues', 'Home_Web_Day7'
]]

In [553]:
engine_unmatched.shape

(464, 32)

In [554]:
internal_unmatched.shape

(388, 31)

In [555]:
exact_merged[['IMPRESSIONS', 'SPEND','CLICKS','CONVERSIONS']] = 0

In [556]:
Search_Phone = pd.concat([exact_merged, internal_unmatched], ignore_index=True)

In [557]:
Search_Phone = Search_Phone.fillna(0)

/tmp/ipykernel_1887463/2545590212.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Search_Phone = Search_Phone.fillna(0)


In [558]:
Search_Phone.shape

(1166, 31)

In [559]:
Search_Web.shape

(1354, 31)

In [560]:
Search_report = pd.concat([Search_Web, Search_Phone], ignore_index=True)

In [594]:
Search_report = Search_report.fillna(0)

In [595]:
# Add new column based on logic
def classify(value):
    value = str(value).upper()
    if "AUTO" in value:
        return "Auto"
    elif "HOME" in value or "CONDO" in value:
        return "Home"
    elif "ENTERPRISE" in value:
        return "Enterprise"
    else:
        return ""
Search_report['Category'] = Search_report['CAMPAIGN_NAME'].apply(classify)


In [596]:
Search_report.lead_monyr.min()

'2025-01'

In [597]:
Search_report.head()

,lead_monyr,CAMPAIGN_NAME,keyword_type,Engine,IMPRESSIONS,SPEND,CLICKS,CONVERSIONS,lead,quote,issue,day0,day7,Auto_Phone_leads,Auto_Web_leads,Seg_1_2_Auto_Web_leads,Seg_0_Auto_Web_leads,Seg_1_2_Auto_Phone_leads,Seg_0_Auto_Phone_leads,Auto_Issues,Home_Phone_leads,Home_Web_leads,Home_Issues,Auto_Day7,Home_Day7,Auto_Web_Issues,Auto_Phone_Issues,Auto_Web_Day7,Home_Web_Issues,Home_Phone_Issues,Home_Web_Day7,Category
0,2025-01,AMICA AUTO - BRAND,Brand,GOOGLE,14789.0,145271.48,5629.0,1584.51,2004.0,2004.0,59.0,12.0,33.0,0.0,1518.0,728.0,3.0,0.0,0.0,46.0,0.0,486.0,13.0,27.0,6.0,46.0,0.0,27.0,13.0,0.0,0.0,Auto
1,2025-01,AMICA AUTO - BRAND - BING,Brand,BING,2887.0,24502.47,869.0,313.00,370.0,370.0,24.0,2.0,8.0,0.0,278.0,149.0,2.0,0.0,0.0,18.0,0.0,92.0,6.0,7.0,1.0,18.0,0.0,7.0,6.0,0.0,0.0,Auto
2,2025-01,AMICA BRAND - CONDO INSURANCE,Brand,GOOGLE,83.0,1025.64,33.0,8.04,11.0,11.0,0.0,0.0,0.0,0.0,3.0,3.0,0.0,0.0,0.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Home
3,2025-01,AMICA BRAND - ENTERPRISE - NON STRATEGIC - BING,Brand,BING,5995.0,15407.74,1694.0,360.00,532.0,532.0,8.0,0.0,3.0,0.0,311.0,223.0,0.0,0.0,0.0,4.0,0.0,221.0,4.0,1.0,2.0,4.0,0.0,1.0,4.0,0.0,0.0,Enterprise
4,2025-01,AMICA BRAND - ENTERPRISE - NON-STRATEGIC,Brand,GOOGLE,13130.0,42192.42,5008.0,1478.36,1614.0,1614.0,26.0,2.0,5.0,0.0,903.0,641.0,2.0,0.0,0.0,16.0,0.0,711.0,10.0,5.0,0.0,16.0,0.0,5.0,10.0,0.0,0.0,Enterprise


In [598]:
Search_report.rename(columns={
    'lead_monyr': 'Year_Month',
    'CAMPAIGN_NAME': 'Campaign',
    'keyword_type': 'Keyword_type',
    'IMPRESSIONS': 'Impressions',
    'SPEND': 'Spend',
    'CLICKS': 'Clicks',
    'CONVERSIONS' : 'Conversions',
    'lead': 'Lead',
    'quote': 'Quote',
    'issue': 'Issue',
    'day0': 'Day0',
    'day7': 'Day7'
    
}, inplace=True)

In [599]:
Search_report['dev_issues'] = Search_report.apply(lambda row: row['Day7'] / 0.55 if row['Year_Month'] >= '2025-11' else row['Issue'],axis=1)

In [600]:

def classify_campaign(campaign):
    campaign = str(campaign).upper()
    if "AUTO" in campaign:
        return "Auto"
    elif "HOME" in campaign or "CONDO" in campaign:
        return "Home"
    elif "ENTERPRISE" in campaign:
        return "Enterprise"
    else:
        return ""

Search_report['Account'] = Search_report['Campaign'].apply(classify_campaign)

In [601]:
Search_report.to_csv('/data/Users/Shelly/Search Report/Search_Mutual_Report_campaign_level.csv')

### attribution logic

In [568]:
phone_re = pd.read_excel('/data/Users/Shelly/Search_Report/phone_reattribution.xlsx')

In [569]:
phone_re.head()

,Campaign1,Campaign2
0,HOME - CONVERSION TRACKING,AMICA BRAND - HOME INSURANCE
1,HOME GENERIC,HOME GENERIC - AZ
2,HOME GENERIC,HOME GENERIC - CT
3,HOME GENERIC,HOME GENERIC - DC
4,HOME GENERIC,HOME GENERIC - DE


In [570]:
internal_unmatched.head()

,lead_monyr,CAMPAIGN_NAME,keyword_type,Engine,IMPRESSIONS,SPEND,CLICKS,CONVERSIONS,lead,quote,issue,day0,day7,Auto_Phone_leads,Auto_Web_leads,Seg_1_2_Auto_Web_leads,Seg_0_Auto_Web_leads,Seg_1_2_Auto_Phone_leads,Seg_0_Auto_Phone_leads,Auto_Issues,Home_Phone_leads,Home_Web_leads,Home_Issues,Auto_Day7,Home_Day7,Auto_Web_Issues,Auto_Phone_Issues,Auto_Web_Day7,Home_Web_Issues,Home_Phone_Issues,Home_Web_Day7
9,2025-01,AMICA.COM PRODUCT DEVELOPMENT PAID SEARCH - GO...,Generic,GOOGLE,NaN,NaN,NaN,NaN,1.0,1.0,0.0,0.0,0.0,1.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0,0.0,0,0,0.0,0.0
10,2025-01,AMICA.COM PRODUCT DEVELOPMENT PAID SEARCH - GO...,Generic,GOOGLE,NaN,NaN,NaN,NaN,17.0,14.0,2.0,0.0,0.0,3.0,0,0.0,0.0,2.0,1.0,1.0,14.0,0,1.0,0.0,0.0,0,1.0,0,0,1.0,0.0
11,2025-01,AUTO BUNDLE PAID SEARCH CAMPAIGN - BING,Generic,BING,NaN,NaN,NaN,NaN,34.0,29.0,5.0,1.0,1.0,20.0,0,0.0,0.0,12.0,2.0,2.0,14.0,0,3.0,0.0,1.0,0,2.0,0,0,3.0,1.0
12,2025-01,AUTO BUNDLE PAID SEARCH CAMPAIGN - GOOGLE,Generic,GOOGLE,NaN,NaN,NaN,NaN,17.0,13.0,0.0,0.0,0.0,11.0,0,0.0,0.0,7.0,2.0,0.0,6.0,0,0.0,0.0,0.0,0,0.0,0,0,0.0,0.0
18,2025-01,AUTO GENERIC - CT - BUNDLE,Generic,GOOGLE,NaN,NaN,NaN,NaN,6.0,5.0,0.0,0.0,0.0,3.0,0,0.0,0.0,2.0,0.0,0.0,3.0,0,0.0,0.0,0.0,0,0.0,0,0,0.0,0.0


In [571]:
exact_merged.shape

(778, 31)

In [572]:
phone_re.shape

(78, 2)

In [573]:

phone_lookup = pd.merge(
    exact_merged,
    phone_re,
    left_on=['CAMPAIGN_NAME'],
    right_on=['Campaign2'],
    how='left'
)


In [574]:
phone_lookup.columns

Index(['lead_monyr', 'CAMPAIGN_NAME', 'keyword_type', 'Engine', 'IMPRESSIONS',
       'SPEND', 'CLICKS', 'CONVERSIONS', 'lead', 'quote', 'issue', 'day0',
       'day7', 'Auto_Phone_leads', 'Auto_Web_leads', 'Seg_1_2_Auto_Web_leads',
       'Seg_0_Auto_Web_leads', 'Seg_1_2_Auto_Phone_leads',
       'Seg_0_Auto_Phone_leads', 'Auto_Issues', 'Home_Phone_leads',
       'Home_Web_leads', 'Home_Issues', 'Auto_Day7', 'Home_Day7',
       'Auto_Web_Issues', 'Auto_Phone_Issues', 'Auto_Web_Day7',
       'Home_Web_Issues', 'Home_Phone_Issues', 'Home_Web_Day7', 'Campaign1',
       'Campaign2'],
      dtype='object')

In [575]:

import pandas as pd

# columns you want to normalize within each group and overwrite
cols = ['lead', 'quote', 'issue', 'day0',
       'day7', 'Auto_Phone_leads', 'Auto_Web_leads', 'Seg_1_2_Auto_Web_leads',
       'Seg_0_Auto_Web_leads', 'Seg_1_2_Auto_Phone_leads',
       'Seg_0_Auto_Phone_leads', 'Auto_Issues', 'Home_Phone_leads',
       'Home_Web_leads', 'Home_Issues', 'Auto_Day7', 'Home_Day7',
       'Auto_Web_Issues', 'Auto_Phone_Issues', 'Auto_Web_Day7',
       'Home_Web_Issues', 'Home_Phone_Issues', 'Home_Web_Day7']
group_cols = ['lead_monyr', 'Campaign1']

# compute per-group sums for those columns
sums = (
    phone_lookup
    .groupby(group_cols)[cols]
    .transform('sum')
    .replace(0, pd.NA)  # avoid division by zero (optional)
)

# overwrite the original columns with their normalized values
phone_lookup[cols] = phone_lookup[cols].div(sums)

# rename the overwritten columns to add the "_factor" suffix
phone_lookup.rename(columns={c: f"{c}_factor" for c in cols}, inplace=True)

# (optional) if you prefer zeros instead of NA for groups that summed to 0:
phone_lookup[[f"{c}_factor" for c in cols]] = phone_lookup[[f"{c}_factor" for c in cols]].fillna(0)


/tmp/ipykernel_1887463/1005002616.py:28: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  phone_lookup[[f"{c}_factor" for c in cols]] = phone_lookup[[f"{c}_factor" for c in cols]].fillna(0)


In [576]:
phone_lookup.columns

Index(['lead_monyr', 'CAMPAIGN_NAME', 'keyword_type', 'Engine', 'IMPRESSIONS',
       'SPEND', 'CLICKS', 'CONVERSIONS', 'lead_factor', 'quote_factor',
       'issue_factor', 'day0_factor', 'day7_factor', 'Auto_Phone_leads_factor',
       'Auto_Web_leads_factor', 'Seg_1_2_Auto_Web_leads_factor',
       'Seg_0_Auto_Web_leads_factor', 'Seg_1_2_Auto_Phone_leads_factor',
       'Seg_0_Auto_Phone_leads_factor', 'Auto_Issues_factor',
       'Home_Phone_leads_factor', 'Home_Web_leads_factor',
       'Home_Issues_factor', 'Auto_Day7_factor', 'Home_Day7_factor',
       'Auto_Web_Issues_factor', 'Auto_Phone_Issues_factor',
       'Auto_Web_Day7_factor', 'Home_Web_Issues_factor',
       'Home_Phone_Issues_factor', 'Home_Web_Day7_factor', 'Campaign1',
       'Campaign2'],
      dtype='object')

In [577]:

phone_lookup = phone_lookup.drop(columns=['Engine', 'IMPRESSIONS',
       'SPEND', 'CLICKS', 'CONVERSIONS'])


In [578]:
internal_unmatched.columns

Index(['lead_monyr', 'CAMPAIGN_NAME', 'keyword_type', 'Engine', 'IMPRESSIONS',
       'SPEND', 'CLICKS', 'CONVERSIONS', 'lead', 'quote', 'issue', 'day0',
       'day7', 'Auto_Phone_leads', 'Auto_Web_leads', 'Seg_1_2_Auto_Web_leads',
       'Seg_0_Auto_Web_leads', 'Seg_1_2_Auto_Phone_leads',
       'Seg_0_Auto_Phone_leads', 'Auto_Issues', 'Home_Phone_leads',
       'Home_Web_leads', 'Home_Issues', 'Auto_Day7', 'Home_Day7',
       'Auto_Web_Issues', 'Auto_Phone_Issues', 'Auto_Web_Day7',
       'Home_Web_Issues', 'Home_Phone_Issues', 'Home_Web_Day7'],
      dtype='object')

In [579]:

internal_unmatched = internal_unmatched.drop(columns=['Engine', 'IMPRESSIONS',
       'SPEND', 'CLICKS', 'CONVERSIONS'])

In [580]:

reattributed = pd.merge(
    phone_lookup,
    internal_unmatched,
    left_on=['lead_monyr','Campaign1'],
    right_on=['lead_monyr','CAMPAIGN_NAME'],
    how='left'
)


In [581]:
reattributed.columns

Index(['lead_monyr', 'CAMPAIGN_NAME_x', 'keyword_type_x', 'lead_factor',
       'quote_factor', 'issue_factor', 'day0_factor', 'day7_factor',
       'Auto_Phone_leads_factor', 'Auto_Web_leads_factor',
       'Seg_1_2_Auto_Web_leads_factor', 'Seg_0_Auto_Web_leads_factor',
       'Seg_1_2_Auto_Phone_leads_factor', 'Seg_0_Auto_Phone_leads_factor',
       'Auto_Issues_factor', 'Home_Phone_leads_factor',
       'Home_Web_leads_factor', 'Home_Issues_factor', 'Auto_Day7_factor',
       'Home_Day7_factor', 'Auto_Web_Issues_factor',
       'Auto_Phone_Issues_factor', 'Auto_Web_Day7_factor',
       'Home_Web_Issues_factor', 'Home_Phone_Issues_factor',
       'Home_Web_Day7_factor', 'Campaign1', 'Campaign2', 'CAMPAIGN_NAME_y',
       'keyword_type_y', 'lead', 'quote', 'issue', 'day0', 'day7',
       'Auto_Phone_leads', 'Auto_Web_leads', 'Seg_1_2_Auto_Web_leads',
       'Seg_0_Auto_Web_leads', 'Seg_1_2_Auto_Phone_leads',
       'Seg_0_Auto_Phone_leads', 'Auto_Issues', 'Home_Phone_leads',
       'H

In [582]:
reattributed.head()

,lead_monyr,CAMPAIGN_NAME_x,keyword_type_x,lead_factor,quote_factor,issue_factor,day0_factor,day7_factor,Auto_Phone_leads_factor,Auto_Web_leads_factor,Seg_1_2_Auto_Web_leads_factor,Seg_0_Auto_Web_leads_factor,Seg_1_2_Auto_Phone_leads_factor,Seg_0_Auto_Phone_leads_factor,Auto_Issues_factor,Home_Phone_leads_factor,Home_Web_leads_factor,Home_Issues_factor,Auto_Day7_factor,Home_Day7_factor,Auto_Web_Issues_factor,Auto_Phone_Issues_factor,Auto_Web_Day7_factor,Home_Web_Issues_factor,Home_Phone_Issues_factor,Home_Web_Day7_factor,Campaign1,Campaign2,CAMPAIGN_NAME_y,keyword_type_y,lead,quote,issue,day0,day7,Auto_Phone_leads,Auto_Web_leads,Seg_1_2_Auto_Web_leads,Seg_0_Auto_Web_leads,Seg_1_2_Auto_Phone_leads,Seg_0_Auto_Phone_leads,Auto_Issues,Home_Phone_leads,Home_Web_leads,Home_Issues,Auto_Day7,Home_Day7,Auto_Web_Issues,Auto_Phone_Issues,Auto_Web_Day7,Home_Web_Issues,Home_Phone_Issues,Home_Web_Day7
0,2025-01,AMICA AUTO - BRAND,Brand,1.0,1.0,1.0,1.0,1.0,1.0,0,0,0,1.0,1.0,1.0,1.0,0,1.0,1.0,1.0,0,1.0,0,0,1.0,1.0,GOOGLE AUTO CALL EXTENSION - CONVERSION TRACKING,AMICA AUTO - BRAND,GOOGLE AUTO CALL EXTENSION - CONVERSION TRACKING,Generic,4.0,3.0,2.0,0.0,2.0,3.0,0,0.0,0.0,1.0,1.0,2.0,1.0,0,0.0,2.0,0.0,0,2.0,0,0,0.0,0.0
1,2025-01,AMICA AUTO - BRAND - BING,Brand,1.0,1.0,1.0,1.0,1.0,1.0,0,0,0,1.0,1.0,1.0,1.0,0,1.0,1.0,1.0,0,1.0,0,0,1.0,1.0,BING AUTO - CONVERSION TRACKING,AMICA AUTO - BRAND - BING,BING AUTO - CONVERSION TRACKING,Generic,2.0,2.0,0.0,0.0,0.0,1.0,0,0.0,0.0,0.0,0.0,0.0,1.0,0,0.0,0.0,0.0,0,0.0,0,0,0.0,0.0
2,2025-01,AMICA BRAND - ENTERPRISE - NON STRATEGIC - BING,Brand,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0,0.0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-01,AMICA BRAND - ENTERPRISE - NON-STRATEGIC,Brand,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0,0.0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-01,AMICA BRAND - ENTERPRISE - STRATEGIC,Brand,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0,0.0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [583]:

import pandas as pd

# ---- 1) Define the columns you mentioned ----
factor_cols = [
    'lead_factor', 'quote_factor', 'issue_factor', 'day0_factor', 'day7_factor',
    'Auto_Phone_leads_factor', 'Auto_Web_leads_factor',
    'Seg_1_2_Auto_Web_leads_factor', 'Seg_0_Auto_Web_leads_factor',
    'Seg_1_2_Auto_Phone_leads_factor', 'Seg_0_Auto_Phone_leads_factor',
    'Auto_Issues_factor', 'Home_Phone_leads_factor',
    'Home_Web_leads_factor', 'Home_Issues_factor', 'Auto_Day7_factor',
    'Home_Day7_factor', 'Auto_Web_Issues_factor',
    'Auto_Phone_Issues_factor', 'Auto_Web_Day7_factor',
    'Home_Web_Issues_factor', 'Home_Phone_Issues_factor',
    'Home_Web_Day7_factor'
]

base_cols = [
    'lead', 'quote', 'issue', 'day0', 'day7',
    'Auto_Phone_leads', 'Auto_Web_leads', 'Seg_1_2_Auto_Web_leads',
    'Seg_0_Auto_Web_leads', 'Seg_1_2_Auto_Phone_leads',
    'Seg_0_Auto_Phone_leads', 'Auto_Issues', 'Home_Phone_leads',
    'Home_Web_leads', 'Home_Issues', 'Auto_Day7', 'Home_Day7','Auto_Web_Issues','Auto_Phone_Issues','Auto_Web_Day7',
    'Home_Web_Issues',	'Home_Phone_Issues','Home_Web_Day7'

]

# ---- 2) Build (base, factor) pairs where both columns actually exist ----
pairs = []
for f in factor_cols:
    if f.endswith('_factor'):
        base = f[:-7]  # strip the "_factor" suffix
        if base in reattributed.columns and f in reattributed.columns:
            pairs.append((base, f))

# If you only want to compute for the explicitly provided base_cols, filter to those:
# pairs = [(b, f) for (b, f) in pairs if b in set(base_cols)]

# ---- 3) Create the new columns (UPPERCASE names) as base * factor ----
# Notes:
# - If you want NaNs where either side is missing, this is fine as-is.
# - If you want to treat missing factor as 0, you could fillna on the factor part first.
for base, factor in pairs:
    new_col = base.upper()  # e.g. 'lead' -> 'LEAD'
    reattributed[new_col] = reattributed[base] * reattributed[factor].round(2)

# ---- 4) Drop all original base and factor columns you listed ----
# Safely intersect with existing columns so drop() won’t error.
to_drop = list(set(factor_cols + base_cols) & set(reattributed.columns))
reattributed.drop(columns=to_drop, inplace=True)

# (Optional) If you want missing products to be 0 instead of NaN:
product_cols = [b.upper() for (b, f) in pairs]
reattributed[product_cols] = reattributed[product_cols].fillna(0).round(2)


/tmp/ipykernel_1887463/3570333837.py:53: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  reattributed[product_cols] = reattributed[product_cols].fillna(0).round(2)


In [585]:
reattributed.LEAD.sum()

3482.25

In [586]:
internal_unmatched.lead.sum()

5463.0

In [587]:
reattributed.head()

,lead_monyr,CAMPAIGN_NAME_x,keyword_type_x,Campaign1,Campaign2,CAMPAIGN_NAME_y,keyword_type_y,LEAD,QUOTE,ISSUE,DAY0,DAY7,AUTO_PHONE_LEADS,AUTO_WEB_LEADS,SEG_1_2_AUTO_WEB_LEADS,SEG_0_AUTO_WEB_LEADS,SEG_1_2_AUTO_PHONE_LEADS,SEG_0_AUTO_PHONE_LEADS,AUTO_ISSUES,HOME_PHONE_LEADS,HOME_WEB_LEADS,HOME_ISSUES,AUTO_DAY7,HOME_DAY7,AUTO_WEB_ISSUES,AUTO_PHONE_ISSUES,AUTO_WEB_DAY7,HOME_WEB_ISSUES,HOME_PHONE_ISSUES,HOME_WEB_DAY7
0,2025-01,AMICA AUTO - BRAND,Brand,GOOGLE AUTO CALL EXTENSION - CONVERSION TRACKING,AMICA AUTO - BRAND,GOOGLE AUTO CALL EXTENSION - CONVERSION TRACKING,Generic,4.0,3.0,2.0,0.0,2.0,3.0,0,0.0,0.0,1.0,1.0,2.0,1.0,0,0.0,2.0,0.0,0,2.0,0,0,0.0,0.0
1,2025-01,AMICA AUTO - BRAND - BING,Brand,BING AUTO - CONVERSION TRACKING,AMICA AUTO - BRAND - BING,BING AUTO - CONVERSION TRACKING,Generic,2.0,2.0,0.0,0.0,0.0,1.0,0,0.0,0.0,0.0,0.0,0.0,1.0,0,0.0,0.0,0.0,0,0.0,0,0,0.0,0.0
2,2025-01,AMICA BRAND - ENTERPRISE - NON STRATEGIC - BING,Brand,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0,0.0,0,0,0.0,0.0
3,2025-01,AMICA BRAND - ENTERPRISE - NON-STRATEGIC,Brand,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0,0.0,0,0,0.0,0.0
4,2025-01,AMICA BRAND - ENTERPRISE - STRATEGIC,Brand,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0,0.0,0,0,0.0,0.0


In [228]:
#internal_unmatched_filtered.to_csv('/data/Users/Shelly/Search Report/internal_unmatched_filtered.csv')

In [588]:

to_remove = {
    'HOME - CONVERSION TRACKING',
    'HOME GENERIC',
    'GOOGLE AUTO GENERIC - CONVERSION TRACKING',
    'HOME CALL EXTENSION - CONVERSION TRACKING',
    'BING AUTO - CONVERSION TRACKING',
    'AMICA.COM PRODUCT DEVELOPMENT PAID SEARCH - GOOGLE (HOME)',
    'GOOGLE AUTO CALL EXTENSION - CONVERSION TRACKING',
    'PMAX TEST PILOT',
    'AUTO GENERIC - CT - BUNDLE'  # fixed closing quote here
}

# Keep only rows where campaign is NOT in the list
internal_unmatched_filtered = internal_unmatched[~internal_unmatched['CAMPAIGN_NAME'].isin(to_remove)]

reattributed_filtered = reattributed[reattributed['Campaign1'].isin(to_remove)]

In [589]:
Search_Phone = pd.concat([exact_merged, internal_unmatched_filtered], ignore_index=True)

In [590]:
Search_report = pd.concat([Search_Web, Search_Phone], ignore_index=True)

In [591]:
Search_report = Search_report.fillna(0)

/tmp/ipykernel_1887463/452573369.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Search_report = Search_report.fillna(0)


In [592]:
reattributed_filtered.to_csv('/data/Users/Shelly/Search Report/reattributed_filtered.csv')

In [409]:
reattributed_filtered.head()

,lead_monyr,CAMPAIGN_NAME_x,keyword_type_x,Campaign1,Campaign2,CAMPAIGN_NAME_y,keyword_type_y,LEAD,QUOTE,ISSUE,DAY0,DAY7,AUTO_PHONE_LEADS,AUTO_WEB_LEADS,SEG_1_2_AUTO_WEB_LEADS,SEG_0_AUTO_WEB_LEADS,SEG_1_2_AUTO_PHONE_LEADS,SEG_0_AUTO_PHONE_LEADS,AUTO_ISSUES,HOME_PHONE_LEADS,HOME_WEB_LEADS,HOME_ISSUES,AUTO_DAY7,HOME_DAY7,AUTO_WEB_ISSUES,AUTO_PHONE_ISSUES,AUTO_WEB_DAY7,HOME_WEB_ISSUES,HOME_PHONE_ISSUES,HOME_WEB_DAY7
0,2025-01,AMICA AUTO - BRAND,Brand,GOOGLE AUTO CALL EXTENSION - CONVERSION TRACKING,AMICA AUTO - BRAND,GOOGLE AUTO CALL EXTENSION - CONVERSION TRACKING,Generic,4.0,3.00,2.0,0.0,2.0,3.0,0,0.0,0.0,1.00,1.00,2.0,1.0,0,0.0,2.0,0.0,0,2.0,0,0,0.0,0.0
1,2025-01,AMICA AUTO - BRAND - BING,Brand,BING AUTO - CONVERSION TRACKING,AMICA AUTO - BRAND - BING,BING AUTO - CONVERSION TRACKING,Generic,2.0,2.00,0.0,0.0,0.0,1.0,0,0.0,0.0,0.00,0.00,0.0,1.0,0,0.0,0.0,0.0,0,0.0,0,0,0.0,0.0
6,2025-01,AMICA BRAND - HOME INSURANCE,Brand,HOME - CONVERSION TRACKING,AMICA BRAND - HOME INSURANCE,HOME - CONVERSION TRACKING,Generic,76.0,56.00,3.0,0.0,3.0,13.0,0,0.0,0.0,9.00,0.00,1.0,63.0,0,2.0,1.0,2.0,0,1.0,0,0,2.0,2.0
8,2025-01,AUTO GENERIC - AZ,Generic,GOOGLE AUTO GENERIC - CONVERSION TRACKING,AUTO GENERIC - AZ,GOOGLE AUTO GENERIC - CONVERSION TRACKING,Generic,1.0,0.70,0.0,0.0,0.0,0.9,0,0.0,0.0,0.11,0.24,0.0,0.0,0,0.0,0.0,0.0,0,0.0,0,0,0.0,0.0
9,2025-01,AUTO GENERIC - CT,Generic,GOOGLE AUTO GENERIC - CONVERSION TRACKING,AUTO GENERIC - CT,GOOGLE AUTO GENERIC - CONVERSION TRACKING,Generic,1.0,0.77,0.0,0.0,0.0,0.9,0,0.0,0.0,0.08,0.16,0.0,0.0,0,0.0,0.0,0.0,0,0.0,0,0,0.0,0.0


In [593]:
reattributed_filtered.AUTO_PHONE_LEADS.sum()

1012.0500000000002

#### Test

In [421]:
internal_unmatched.to_csv('/data/Users/Shelly/Search Report/internal_unmatched_test.csv')

In [987]:
exact_merged.to_csv('/data/Users/Shelly/Search Report/phone_matched.csv')

In [979]:
engine.head()

,lead_monyr,CAMPAIGN_NAME,keyword_type,Engine,IMPRESSIONS,SPEND,CLICKS,CONVERSIONS
0,2025-01,AMICA AUTO - BRAND,Brand,GOOGLE,14789,145271.48,5629,1584.51
1,2025-01,AMICA AUTO - BRAND - BING,Brand,BING,2887,24502.47,869,313.00
2,2025-01,AMICA BRAND - CONDO INSURANCE,Brand,GOOGLE,83,1025.64,33,8.04
3,2025-01,AMICA BRAND - ENTERPRISE - NON STRATEGIC - BING,Brand,BING,5995,15407.74,1694,360.00
4,2025-01,AMICA BRAND - ENTERPRISE - NON-STRATEGIC,Brand,GOOGLE,13130,42192.42,5008,1478.36


In [980]:
internal_grouped.head()

,lead_monyr,campaign,keyword_type,Engine,lead,quote,issue,day0,day7,Auto_Phone_leads,Auto_Web_leads,Seg_1_2_Auto_Web_leads,Seg_0_Auto_Web_leads,Seg_1_2_Auto_Phone_leads,Seg_0_Auto_Phone_leads,Auto_Issues,Home_Phone_leads,Home_Web_leads,Home_Issues,Auto_Day7,Home_Day7,Auto_Web_Issues,Auto_Phone_Issues,Auto_Web_Day7,Home_Web_Issues,Home_Phone_Issues,Home_Web_Day7
0,2025-01,AMICA AUTO - BRAND,Brand,GOOGLE,337,288,54,19.0,37.0,266.0,0,0.0,0,113.0,39,36.0,71.0,0,18.0,24.0,13.0,0,36.0,0,0,18.0,13.0
1,2025-01,AMICA AUTO - BRAND - BING,Brand,BING,26,22,7,2.0,3.0,20.0,0,0.0,0,9.0,2,4.0,6.0,0,3.0,2.0,1.0,0,4.0,0,0,3.0,1.0
2,2025-01,AMICA BRAND - ENTERPRISE - NON STRATEGIC - BING,Brand,BING,92,83,7,3.0,6.0,50.0,0,0.0,0,32.0,1,2.0,42.0,0,5.0,2.0,4.0,0,2.0,0,0,5.0,4.0
3,2025-01,AMICA BRAND - ENTERPRISE - NON-STRATEGIC,Brand,GOOGLE,337,274,25,4.0,11.0,187.0,0,0.0,0,97.0,27,14.0,150.0,0,11.0,6.0,5.0,0,14.0,0,0,11.0,5.0
4,2025-01,AMICA BRAND - ENTERPRISE - STRATEGIC,Brand,GOOGLE,828,702,119,39.0,80.0,556.0,0,0.0,0,234.0,78,59.0,272.0,0,60.0,44.0,36.0,0,59.0,0,0,60.0,36.0


In [984]:
internal_unmatched.head()

,lead_monyr,CAMPAIGN_NAME,keyword_type,Engine,IMPRESSIONS,SPEND,CLICKS,CONVERSIONS,lead,quote,issue,day0,day7,Auto_Phone_leads,Auto_Web_leads,Seg_1_2_Auto_Web_leads,Seg_0_Auto_Web_leads,Seg_1_2_Auto_Phone_leads,Seg_0_Auto_Phone_leads,Auto_Issues,Home_Phone_leads,Home_Web_leads,Home_Issues,Auto_Day7,Home_Day7
9,2025-01,AMICA.COM PRODUCT DEVELOPMENT PAID SEARCH - GO...,Generic,GOOGLE,NaN,NaN,NaN,NaN,1.0,1.0,0.0,0.0,0.0,1.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0
10,2025-01,AMICA.COM PRODUCT DEVELOPMENT PAID SEARCH - GO...,Generic,GOOGLE,NaN,NaN,NaN,NaN,17.0,14.0,2.0,0.0,0.0,3.0,0,0.0,0.0,2.0,1.0,1.0,14.0,0,1.0,0.0,0.0
11,2025-01,AUTO BUNDLE PAID SEARCH CAMPAIGN - BING,Generic,BING,NaN,NaN,NaN,NaN,35.0,30.0,5.0,1.0,1.0,21.0,0,0.0,0.0,13.0,2.0,2.0,14.0,0,3.0,0.0,1.0
12,2025-01,AUTO BUNDLE PAID SEARCH CAMPAIGN - GOOGLE,Generic,GOOGLE,NaN,NaN,NaN,NaN,17.0,13.0,0.0,0.0,0.0,11.0,0,0.0,0.0,7.0,2.0,0.0,6.0,0,0.0,0.0,0.0
18,2025-01,AUTO GENERIC - CT - BUNDLE,Generic,GOOGLE,NaN,NaN,NaN,NaN,6.0,4.0,0.0,0.0,0.0,3.0,0,0.0,0.0,2.0,0.0,0.0,3.0,0,0.0,0.0,0.0


In [1300]:
reattribute = internal_unmatched.groupby(['CAMPAIGN_NAME'], as_index=False).agg({
    'lead': 'sum'
}).sort_values(by='lead', ascending=False)

engine.groupby(['lead_monyr','CAMPAIGN_NAME','Engine'], as_index=False).agg({
    'CONVERSIONS': 'sum'
}).sort_values(by='CONVERSIONS', ascending=False)

internal_grouped.groupby(['lead_monyr','campaign','Engine'], as_index=False).agg({
    'lead': 'sum'
}).sort_values(by='lead', ascending=False)

In [1199]:
engine.head()

,lead_monyr,CAMPAIGN_NAME,keyword_type,Engine,IMPRESSIONS,SPEND,CLICKS,CONVERSIONS
0,2025-01,AMICA AUTO - BRAND,Brand,GOOGLE,14789,145271.48,5629,1584.51
1,2025-01,AMICA AUTO - BRAND - BING,Brand,BING,2887,24502.47,869,313.00
2,2025-01,AMICA BRAND - CONDO INSURANCE,Brand,GOOGLE,83,1025.64,33,8.04
3,2025-01,AMICA BRAND - ENTERPRISE - NON STRATEGIC - BING,Brand,BING,5995,15407.74,1694,360.00
4,2025-01,AMICA BRAND - ENTERPRISE - NON-STRATEGIC,Brand,GOOGLE,13130,42192.42,5008,1478.36


In [1200]:
engine[engine['CAMPAIGN_NAME'] == 'HOME GENERIC - PA - BING'].head()

,lead_monyr,CAMPAIGN_NAME,keyword_type,Engine,IMPRESSIONS,SPEND,CLICKS,CONVERSIONS
74,2025-01,HOME GENERIC - PA - BING,Generic,BING,21947,13022.05,408,99.0
163,2025-02,HOME GENERIC - PA - BING,Generic,BING,20364,13031.93,350,69.0
254,2025-03,HOME GENERIC - PA - BING,Generic,BING,27747,25985.72,609,102.0
348,2025-04,HOME GENERIC - PA - BING,Generic,BING,20022,23075.91,444,96.0
442,2025-05,HOME GENERIC - PA - BING,Generic,BING,19323,24366.50,375,92.0


In [1202]:
internal_grouped[internal_grouped['campaign'] == 'HOME GENERIC - PA - BING'].Auto_Phone_leads.sum()

28.0

In [1201]:
internal_grouped[internal_grouped['campaign'] == 'HOME GENERIC - PA - BING'].head()

,lead_monyr,campaign,keyword_type,Engine,lead,quote,issue,day0,day7,Auto_Phone_leads,Auto_Web_leads,Seg_1_2_Auto_Web_leads,Seg_0_Auto_Web_leads,Seg_1_2_Auto_Phone_leads,Seg_0_Auto_Phone_leads,Auto_Issues,Home_Phone_leads,Home_Web_leads,Home_Issues,Auto_Day7,Home_Day7,Auto_Web_Issues,Auto_Phone_Issues,Auto_Web_Day7,Home_Web_Issues,Home_Phone_Issues,Home_Web_Day7
101,2025-01,HOME GENERIC - PA - BING,Generic,BING,6,5,2,0.0,0.0,2.0,0,0.0,0,1.0,0,1.0,4.0,0,1.0,0.0,0.0,0,1.0,0,0,1.0,0.0
221,2025-02,HOME GENERIC - PA - BING,Generic,BING,6,4,0,0.0,0.0,1.0,0,0.0,0,1.0,0,0.0,5.0,0,0.0,0.0,0.0,0,0.0,0,0,0.0,0.0
342,2025-03,HOME GENERIC - PA - BING,Generic,BING,7,5,0,0.0,0.0,2.0,0,0.0,0,1.0,0,0.0,5.0,0,0.0,0.0,0.0,0,0.0,0,0,0.0,0.0
471,2025-04,HOME GENERIC - PA - BING,Generic,BING,6,5,2,2.0,2.0,2.0,0,0.0,0,1.0,0,1.0,4.0,0,1.0,1.0,1.0,0,1.0,0,0,1.0,1.0
589,2025-05,HOME GENERIC - PA - BING,Generic,BING,6,4,0,0.0,0.0,0.0,0,0.0,0,0.0,0,0.0,6.0,0,0.0,0.0,0.0,0,0.0,0,0,0.0,0.0


In [375]:
internal_data = pd.read_csv('/data/Projects/lead_policy_counts/aggregated_df_search.csv')

In [376]:
internal_data = internal_data[(internal_data['lead_dt'] >= start_date1) & (internal_data['lead_dt'] <= end_date1)]
internal_data = internal_data[internal_data['last_touch_medium'].isin(['Paid Search', 'cpc'])]

In [377]:
internal_data = internal_data[internal_data['lead_contact_method'] == 'PHONE']

In [378]:
internal_data.head()

,business_date,line_of_business,lead_dt,bus_monyr,lead_contact_method,business_year,trans_contact_method,business_mon,business_day,state_grp,quality_seg,cov_item_type,lead_medium,state_cd,lead_monyr,last_touch_medium,medium_channel_adjusted,source,campaign,lead,quote,issue,prior_month_issue,day0,day7,same_month_issue
287552,2025-01-01,HOMEOWNERS,2025-01-01,2025-01,PHONE,2025,PHONE,1,1,Non-Target,0,DWELLING,Paid Search,TN,2025-01,Paid Search,Paid Search,NaN,Home Generic,1,1,0,0,0.0,0.0,0.0
287553,2025-01-01,HOMEOWNERS,2025-01-01,2025-01,PHONE,2025,PHONE,1,1,Non-Target,0,DWELLING,Paid Search,OK,2025-01,Paid Search,Paid Search,NaN,Home Call Extension - Conversion Tracking,1,1,0,0,0.0,0.0,0.0
287554,2025-01-01,HOMEOWNERS,2025-01-01,2025-01,PHONE,2025,PHONE,1,1,Non-Target,0,DWELLING,Paid Search,MS,2025-01,Paid Search,Paid Search,NaN,Home - Conversion Tracking,1,0,0,0,0.0,0.0,0.0
287555,2025-01-01,HOMEOWNERS,2025-01-01,2025-01,PHONE,2025,PHONE,1,1,Non-Target,0,DWELLING,Paid Search,MS,2025-01,Paid Search,Paid Search,NaN,Enterprise Call Extension - Conversion Tracking,1,1,0,0,0.0,0.0,0.0
287557,2025-01-01,HOMEOWNERS,2025-01-01,2025-01,PHONE,2025,PHONE,1,1,Target,0,DWELLING,Paid Search,NY,2025-01,Paid Search,Paid Search,NaN,Amica Home Brand campaign (Google),1,1,1,0,1.0,1.0,1.0


In [379]:
pd.set_option('display.max_rows', None)

In [380]:
internal_data.head()

,business_date,line_of_business,lead_dt,bus_monyr,lead_contact_method,business_year,trans_contact_method,business_mon,business_day,state_grp,quality_seg,cov_item_type,lead_medium,state_cd,lead_monyr,last_touch_medium,medium_channel_adjusted,source,campaign,lead,quote,issue,prior_month_issue,day0,day7,same_month_issue
287552,2025-01-01,HOMEOWNERS,2025-01-01,2025-01,PHONE,2025,PHONE,1,1,Non-Target,0,DWELLING,Paid Search,TN,2025-01,Paid Search,Paid Search,NaN,Home Generic,1,1,0,0,0.0,0.0,0.0
287553,2025-01-01,HOMEOWNERS,2025-01-01,2025-01,PHONE,2025,PHONE,1,1,Non-Target,0,DWELLING,Paid Search,OK,2025-01,Paid Search,Paid Search,NaN,Home Call Extension - Conversion Tracking,1,1,0,0,0.0,0.0,0.0
287554,2025-01-01,HOMEOWNERS,2025-01-01,2025-01,PHONE,2025,PHONE,1,1,Non-Target,0,DWELLING,Paid Search,MS,2025-01,Paid Search,Paid Search,NaN,Home - Conversion Tracking,1,0,0,0,0.0,0.0,0.0
287555,2025-01-01,HOMEOWNERS,2025-01-01,2025-01,PHONE,2025,PHONE,1,1,Non-Target,0,DWELLING,Paid Search,MS,2025-01,Paid Search,Paid Search,NaN,Enterprise Call Extension - Conversion Tracking,1,1,0,0,0.0,0.0,0.0
287557,2025-01-01,HOMEOWNERS,2025-01-01,2025-01,PHONE,2025,PHONE,1,1,Target,0,DWELLING,Paid Search,NY,2025-01,Paid Search,Paid Search,NaN,Amica Home Brand campaign (Google),1,1,1,0,1.0,1.0,1.0


In [156]:
internal_data[internal_data['campaign'] == 'auto generic - ct - bundle']

In [381]:
internal_data.source.unique()

array([nan, 'google', 'bing'], dtype=object)

In [382]:

import pandas as pd

def update_campaign_name(df: pd.DataFrame) -> pd.DataFrame:
    """
    Update df['campaign'] based on specific rules involving 'campaign' and 'state_cd'.
    Rules are applied in sequence. If a rule does not match, the campaign stays as-is.

    Required columns: 'campaign', 'state_cd'
    """

    # --- Validate required columns ---
    required = {'campaign', 'state_cd'}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    # Work on a copy (order of assignments controls precedence)
    out = df['campaign'].copy()

    # --- Shared state sets ---
    states_ct_az = {'CT', 'AZ'}
    strategic_states = {'AZ', 'CT', 'DC', 'DE', 'MA', 'MD', 'ME', 'NH', 'NM', 'NY', 'OR', 'PA', 'RI', 'VT', 'WA', 'VA'}

    # --- Campaign groups from earlier rules ---
    auto_pdps = {
        'Amica.com Product Development Paid Search - Google (Auto)',
        'Amica.com Product Development Paid Search -Bing (Auto)',  # exact per your spec
    }
    home_pdps = {
        'Amica.com Product Development Paid Search - Google (Home)',
        'Amica.com Product Development Paid Search - Bing (Home)',
    }
    enterprise_campaigns = {
        'Enterprise - Conversion Tracking',
        'Brand - Google - Desktop',
        'Enterprise Paid Search Site-desktop only',
        'Enterprise Call Extension - Conversion Tracking',
        'Paid Search Bundle Message Test - Convenience message',
    }
    auto_conv = {
        'Google Auto Generic - Conversion Tracking',
        'Google Auto Call Extension - Conversion Tracking',
        'Bing Auto - Conversion Tracking',
        'Bing Auto Call Extension - Conversion Tracking',
    }
    home_conv = {
        'Home - Conversion Tracking',
        'Home Generic',
        'Bing Home - Conversion Tracking',
        'Bing Home Call Extension - Conversion Tracking',
        'Home Call Extension - Conversion Tracking',
    }

    # --- New campaign names (previous additions) ---
    auto_bundle_google = 'Auto Bundle Paid Search Campaign - Google'
    auto_bundle_bing   = 'Auto Bundle Paid Search Campaign - Bing'
    pmax_home_campaign = 'PMAX Home Campaign - Test States NH, CT, RI'
    bing_enterprise_campaigns = {
        'Bing Enterprise Call Extension - Conversion Tracking',
        'Bing Enterprise - Conversion Tracking',
    }

    # ============================================================
    # 1) PDPs (Auto): CT/AZ -> AUTO GENERIC - {state}
    # ============================================================
    m1 = df['campaign'].isin(auto_pdps) & df['state_cd'].isin(states_ct_az)
    out.loc[m1] = 'AUTO GENERIC - ' + df.loc[m1, 'state_cd']

    # ============================================================
    # 2) PDPs (Home): CT/AZ -> HOME GENERIC - {state}
    # ============================================================
    m2 = df['campaign'].isin(home_pdps) & df['state_cd'].isin(states_ct_az)
    out.loc[m2] = 'HOME GENERIC - ' + df.loc[m2, 'state_cd']

    # ============================================================
    # 3) Enterprise (non-Bing)
    # ============================================================
    m3 = df['campaign'].isin(enterprise_campaigns)
    m3_strat = m3 & df['state_cd'].isin(strategic_states)
    m3_non   = m3 & ~df['state_cd'].isin(strategic_states)
    out.loc[m3_strat] = 'Amica Brand - Enteprise - Strategic'
    out.loc[m3_non]   = 'Amica Brand - Enteprise - Non Strategic'

    # ============================================================
    # 4) Auto conversion tracking -> AUTO GENERIC - {state}
    # ============================================================
    m4 = df['campaign'].isin(auto_conv) & df['state_cd'].isin(strategic_states)
    out.loc[m4] = 'AUTO GENERIC - ' + df.loc[m4, 'state_cd']

    # ============================================================
    # 5) Home conversion tracking -> HOME GENERIC - {state}
    # ============================================================
    m5 = df['campaign'].isin(home_conv) & df['state_cd'].isin(strategic_states)
    out.loc[m5] = 'HOME GENERIC - ' + df.loc[m5, 'state_cd']

    # ============================================================
    # 6) Auto Bundle (Google): PA/NH/RI/CT -> Auto Generic - {state} - Bundle
    # ============================================================
    m6 = (df['campaign'] == auto_bundle_google) & df['state_cd'].isin({'PA', 'NH', 'RI', 'CT'})
    out.loc[m6] = 'Auto Generic - ' + df.loc[m6, 'state_cd'] + ' - Bundle'

    # ============================================================
    # 7) Auto Bundle (Bing):
    #    PA/CT -> Auto Generic - {state} - Bundle
    #    NH/RI -> Auto Generic - NH | RI - Bundle
    # ============================================================
    m7a = (df['campaign'] == auto_bundle_bing) & df['state_cd'].isin({'PA', 'CT'})
    out.loc[m7a] = 'Auto Generic - ' + df.loc[m7a, 'state_cd'] + ' - Bundle'

    m7b = (df['campaign'] == auto_bundle_bing) & df['state_cd'].isin({'NH', 'RI'})
    out.loc[m7b] = 'Auto Generic - NH | RI - Bundle'

    # ============================================================
    # 8) PMAX Home campaign:
    #    NH/MA/VT -> Home Generic - {state}- Performance Max (no space before hyphen)
    # ============================================================
    m8 = (df['campaign'] == pmax_home_campaign) & df['state_cd'].isin({'NH', 'MA', 'VT'})
    out.loc[m8] = 'Home Generic - ' + df.loc[m8, 'state_cd'] + '- Performance Max'

    # ============================================================
    # 9) Bing Enterprise group:
    #    strategic -> Amica Brand - Enteprise - Strategic - Bing
    #    else      -> Amica Brand - Enteprise - Non Strategic - Bing
    # ============================================================
    m9 = df['campaign'].isin(bing_enterprise_campaigns)
    m9_strat = m9 & df['state_cd'].isin(strategic_states)
    m9_non   = m9 & ~df['state_cd'].isin(strategic_states)
    out.loc[m9_strat] = 'Amica Brand - Enteprise - Strategic - Bing'
    out.loc[m9_non]   = 'Amica Brand - Enteprise - Non Strategic - Bing'

    # ============================================================
    # 10) NEW: Brand renames (Google/Bing)
    # ============================================================
    m10a = (df['campaign'] == 'Amica Auto Brand campaign (Google)')
    out.loc[m10a] = 'Amica Auto - Brand'

    m10b = (df['campaign'] == 'Amica Home Brand campaign (Google)')
    out.loc[m10b] = 'Amica Brand - Home Insurance'

    m10c = (df['campaign'] == 'Amica Auto Brand Campaign (Bing)')
    out.loc[m10c] = 'Amica Auto - Brand - Bing'

    m10d = (df['campaign'] == 'Amica Home Brand Campaign (Bing)')
    out.loc[m10d] = 'Amica Brand - Home Insurance - Bing'

    # ============================================================
    # 11) NEW: Pmax Test Pilot -> Auto Generic - Performance Max
    # ============================================================
    m11 = (df['campaign'] == 'Pmax Test Pilot')
    out.loc[m11] = 'Auto Generic - Performance Max'

    # ============================================================
    # 12) NEW: Condo Call Extension - Conversion Tracking -> Condo Insurance - Generic
    # ============================================================
    m12 = (df['campaign'] == 'Condo Call Extension - Conversion Tracking')
    out.loc[m12] = 'Condo Insurance - Generic'

    # --- Write back and return ---
    df = df.copy()
    df['campaign'] = out
    return df



In [383]:
internal_data = update_campaign_name(internal_data)

In [353]:

test_combo = internal_data.groupby(['campaign','state_cd'], as_index=False).agg({
    'lead': 'sum'
}).sort_values(by='lead', ascending=False)


In [356]:
pd.set_option('display.max_rows', 1000)

In [358]:
test_combo[test_combo['campaign'] == 'Home - Conversion Tracking']

,campaign,state_cd,lead
762,Home - Conversion Tracking,NC,122
752,Home - Conversion Tracking,IL,98
766,Home - Conversion Tracking,OH,86
770,Home - Conversion Tracking,TN,60
759,Home - Conversion Tracking,MO,48
755,Home - Conversion Tracking,KY,45
744,Home - Conversion Tracking,AL,44
764,Home - Conversion Tracking,NJ,40
771,Home - Conversion Tracking,TX,38
760,Home - Conversion Tracking,MS,37


internal_data.groupby(['campaign'], as_index=False).agg({
    'lead': 'sum'
}).sort_values(by='lead', ascending=False)


In [322]:
test_combo.to_csv('/data/Users/Shelly/Search Report/campaign_combo.csv')

In [ ]:
test = internal_data[
    (internal_data['lead_contact_method'] == 'PHONE') &
    (internal_data['line_of_business'] == 'HOMEOWNERS')
]

In [ ]:
internal_data[
    (internal_data['lead_contact_method'] == 'PHONE') &
    (internal_data['line_of_business'] == 'HOMEOWNERS')
]['source'].value_counts()

In [ ]:
internal_data[
    (internal_data['lead_contact_method'] == 'PHONE') &
    (internal_data['line_of_business'] == 'HOMEOWNERS')
]['campaign'].value_counts()

In [ ]:
internal_data.shape

In [ ]:
internal_data.groupby(['lead_contact_method', 'source'], as_index=False).agg({
    'issue': 'sum',

})

In [ ]:
google_data.head()

In [ ]:
pd.set_option('display.max_rows', None)

In [ ]:
google_data.groupby(['CAMPAIGN_NAME'], as_index=False).agg({
    'SPEND': 'sum',})

In [ ]:
internal_data.head()

In [ ]:
test = internal_data[internal_data['Home_Phone_leads'] == 1]
test1 = internal_data[internal_data['Auto_Phone_leads'] == 1]

In [ ]:
test.head()

In [ ]:
test.campaign.unique()

In [ ]:
test.source.unique()

In [ ]:
test1.source.unique()

In [ ]:
test1.campaign.unique(
)

In [ ]:
engine.head()

### search nonbrand data pull

In [283]:
internal_data = pd.read_csv('/data/Projects/lead_policy_counts/aggregated_df_search.csv')

In [284]:
internal_data.head()

,business_date,line_of_business,lead_dt,bus_monyr,lead_contact_method,business_year,trans_contact_method,business_mon,business_day,state_grp,quality_seg,cov_item_type,lead_medium,state_cd,lead_monyr,last_touch_medium,medium_channel_adjusted,source,campaign,lead,quote,issue,prior_month_issue,day0,day7,same_month_issue
0,2024-01-01,AUTOMOBILE,2024-01-01,2024-01,WEB,2024,WEB,1,1,Tier2,5,PRIVATE PASSENGER,Affiliates and Aggregators,WA,2024-01,paid,Affiliates and Aggregators,ca,ca_auto_insurance,3,3,0,0,0.0,0.0,0.0
1,2024-01-01,AUTOMOBILE,2024-01-01,2024-01,WEB,2024,WEB,1,1,Tier2,5,PRIVATE PASSENGER,Paid Search,AZ,2024-01,cpc,Paid Search,google,auto generic - t3 - az,1,1,0,0,0.0,0.0,0.0
2,2024-01-01,AUTOMOBILE,2024-01-01,2024-01,WEB,2024,WEB,1,1,Tier2,5,PRIVATE PASSENGER,Paid Search,AZ,2024-01,unknown,Paid Search,NaN,Brand,1,1,0,0,0.0,0.0,0.0
3,2024-01-01,AUTOMOBILE,2024-01-01,2024-01,WEB,2024,WEB,1,1,Tier2,5,PRIVATE PASSENGER,Paid Search,DE,2024-01,cpc,Paid Search,google,auto generic - t3 - de,1,1,0,0,0.0,0.0,0.0
4,2024-01-01,AUTOMOBILE,2024-01-01,2024-01,WEB,2024,WEB,1,1,Tier2,5,PRIVATE PASSENGER,Paid Search,NC,2024-01,cpc,Paid Search,bing,amica brand - enterprise - t1 - pp,1,1,0,0,0.0,0.0,0.0


In [285]:

internal_data["web_to_phone_flag"] = (
    (internal_data["lead_contact_method"] == "WEB") & (internal_data["trans_contact_method"] == "PHONE")
).astype(int)


In [286]:
internal_data["web_to_phone_quote"] = internal_data["quote"].where((internal_data['lead_contact_method'] == "WEB") & (internal_data['trans_contact_method'] == "PHONE"), other=0)

In [287]:
internal_data.head()

,business_date,line_of_business,lead_dt,bus_monyr,lead_contact_method,business_year,trans_contact_method,business_mon,business_day,state_grp,quality_seg,cov_item_type,lead_medium,state_cd,lead_monyr,last_touch_medium,medium_channel_adjusted,source,campaign,lead,quote,issue,prior_month_issue,day0,day7,same_month_issue,web_to_phone_flag,web_to_phone_quote
0,2024-01-01,AUTOMOBILE,2024-01-01,2024-01,WEB,2024,WEB,1,1,Tier2,5,PRIVATE PASSENGER,Affiliates and Aggregators,WA,2024-01,paid,Affiliates and Aggregators,ca,ca_auto_insurance,3,3,0,0,0.0,0.0,0.0,0,0
1,2024-01-01,AUTOMOBILE,2024-01-01,2024-01,WEB,2024,WEB,1,1,Tier2,5,PRIVATE PASSENGER,Paid Search,AZ,2024-01,cpc,Paid Search,google,auto generic - t3 - az,1,1,0,0,0.0,0.0,0.0,0,0
2,2024-01-01,AUTOMOBILE,2024-01-01,2024-01,WEB,2024,WEB,1,1,Tier2,5,PRIVATE PASSENGER,Paid Search,AZ,2024-01,unknown,Paid Search,NaN,Brand,1,1,0,0,0.0,0.0,0.0,0,0
3,2024-01-01,AUTOMOBILE,2024-01-01,2024-01,WEB,2024,WEB,1,1,Tier2,5,PRIVATE PASSENGER,Paid Search,DE,2024-01,cpc,Paid Search,google,auto generic - t3 - de,1,1,0,0,0.0,0.0,0.0,0,0
4,2024-01-01,AUTOMOBILE,2024-01-01,2024-01,WEB,2024,WEB,1,1,Tier2,5,PRIVATE PASSENGER,Paid Search,NC,2024-01,cpc,Paid Search,bing,amica brand - enterprise - t1 - pp,1,1,0,0,0.0,0.0,0.0,0,0


In [288]:

# Define date range variables
start_date1 = '2025-01-01'
end_date1 = '2025-12-31'

In [289]:
internal_data = internal_data[(internal_data['lead_dt'] >= start_date1) & (internal_data['lead_dt'] <= end_date1)]
internal_data = internal_data[internal_data['last_touch_medium'].isin(['Paid Search', 'cpc'])]

In [290]:
internal_data.head()

,business_date,line_of_business,lead_dt,bus_monyr,lead_contact_method,business_year,trans_contact_method,business_mon,business_day,state_grp,quality_seg,cov_item_type,lead_medium,state_cd,lead_monyr,last_touch_medium,medium_channel_adjusted,source,campaign,lead,quote,issue,prior_month_issue,day0,day7,same_month_issue,web_to_phone_flag,web_to_phone_quote
287357,2025-01-01,HOMEOWNERS,2025-01-01,2025-01,PHONE,2025,PHONE,1,1,Non-Target,0,DWELLING,Paid Search,MS,2025-01,Paid Search,Paid Search,NaN,Enterprise Call Extension - Conversion Tracking,1,1,0,0,0.0,0.0,0.0,0,0
287358,2025-01-01,HOMEOWNERS,2025-01-01,2025-01,PHONE,2025,PHONE,1,1,Non-Target,0,DWELLING,Paid Search,MS,2025-01,Paid Search,Paid Search,NaN,Home - Conversion Tracking,1,0,0,0,0.0,0.0,0.0,0,0
287359,2025-01-01,HOMEOWNERS,2025-01-01,2025-01,PHONE,2025,PHONE,1,1,Non-Target,0,DWELLING,Paid Search,NJ,2025-01,Paid Search,Paid Search,NaN,Home - Conversion Tracking,1,1,0,0,0.0,0.0,0.0,0,0
287360,2025-01-01,HOMEOWNERS,2025-01-01,2025-01,PHONE,2025,PHONE,1,1,Non-Target,0,DWELLING,Paid Search,OK,2025-01,Paid Search,Paid Search,NaN,Home Call Extension - Conversion Tracking,1,1,0,0,0.0,0.0,0.0,0,0
287361,2025-01-01,HOMEOWNERS,2025-01-01,2025-01,PHONE,2025,PHONE,1,1,Non-Target,0,DWELLING,Paid Search,TN,2025-01,Paid Search,Paid Search,NaN,Home Generic,1,1,0,0,0.0,0.0,0.0,0,0


In [291]:
def classify_keyword(campaign):
    campaign_upper = str(campaign).upper()
    if 'BRAND' in campaign_upper or 'ENTERPRISE' in campaign_upper:
        return 'Brand'
    elif 'MAX' in campaign_upper:
        return 'PMAX'
    else:
        return 'Generic'

internal_data['keyword_type'] = internal_data['campaign'].apply(classify_keyword)

In [292]:
internal_data = internal_data[internal_data['keyword_type'] == 'Generic']

In [293]:
internal_data.head()

,business_date,line_of_business,lead_dt,bus_monyr,lead_contact_method,business_year,trans_contact_method,business_mon,business_day,state_grp,quality_seg,cov_item_type,lead_medium,state_cd,lead_monyr,last_touch_medium,medium_channel_adjusted,source,campaign,lead,quote,issue,prior_month_issue,day0,day7,same_month_issue,web_to_phone_flag,web_to_phone_quote,keyword_type
287358,2025-01-01,HOMEOWNERS,2025-01-01,2025-01,PHONE,2025,PHONE,1,1,Non-Target,0,DWELLING,Paid Search,MS,2025-01,Paid Search,Paid Search,NaN,Home - Conversion Tracking,1,0,0,0,0.0,0.0,0.0,0,0,Generic
287359,2025-01-01,HOMEOWNERS,2025-01-01,2025-01,PHONE,2025,PHONE,1,1,Non-Target,0,DWELLING,Paid Search,NJ,2025-01,Paid Search,Paid Search,NaN,Home - Conversion Tracking,1,1,0,0,0.0,0.0,0.0,0,0,Generic
287360,2025-01-01,HOMEOWNERS,2025-01-01,2025-01,PHONE,2025,PHONE,1,1,Non-Target,0,DWELLING,Paid Search,OK,2025-01,Paid Search,Paid Search,NaN,Home Call Extension - Conversion Tracking,1,1,0,0,0.0,0.0,0.0,0,0,Generic
287361,2025-01-01,HOMEOWNERS,2025-01-01,2025-01,PHONE,2025,PHONE,1,1,Non-Target,0,DWELLING,Paid Search,TN,2025-01,Paid Search,Paid Search,NaN,Home Generic,1,1,0,0,0.0,0.0,0.0,0,0,Generic
287362,2025-01-01,HOMEOWNERS,2025-01-01,2025-01,PHONE,2025,PHONE,1,1,Target,0,DWELLING,Paid Search,MA,2025-01,Paid Search,Paid Search,NaN,Home - Conversion Tracking,1,1,0,0,0.0,0.0,0.0,0,0,Generic


In [294]:
internal_data['line_of_business'].unique()

array(['HOMEOWNERS', 'AUTOMOBILE'], dtype=object)

In [295]:
internal_data['Engine'] = internal_data['source'].apply(lambda x: x if x in ['google', 'bing'] 
                                          else 'google' if x == 'google.com' 
                                          else 'bing' if x == 'bing.com' 
                                          else 'google')

In [296]:
exclude_list = [
    'QS_AUTO_INSURANCE', 'MA_AUTO_INSURANCE', 'MA_HOME_INSURANCE', 'CA_AUTO_INSURANCE',
    'USNEWS_HOME_INSURANCE', 'CNBC', 'CA_HOME_INSURANCE', 'MONEYGROUP_TIER1PUBS',
    'BEST_MONEY_AUTO_INSURANCE', 'MONEY.COM', '2501R'
]

internal_data = internal_data[
    ~internal_data['campaign'].str.upper().isin(exclude_list)
]

In [298]:
def classify_campaign(campaign):
    campaign = str(campaign).upper()
    if "AUTO" in campaign:
        return "Auto"
    elif "HOME" in campaign or "CONDO" in campaign:
        return "Home"
    elif "ENTERPRISE" in campaign:
        return "Enterprise"
    else:
        return ""

internal_data['Account'] = internal_data['campaign'].apply(classify_campaign)

In [373]:
internal_data = internal_data[internal_data['Engine'] == 'google']

In [299]:
internal_data.head()

,business_date,line_of_business,lead_dt,bus_monyr,lead_contact_method,business_year,trans_contact_method,business_mon,business_day,state_grp,quality_seg,cov_item_type,lead_medium,state_cd,lead_monyr,last_touch_medium,medium_channel_adjusted,source,campaign,lead,quote,issue,prior_month_issue,day0,day7,same_month_issue,web_to_phone_flag,web_to_phone_quote,keyword_type,Engine,Account
287358,2025-01-01,HOMEOWNERS,2025-01-01,2025-01,PHONE,2025,PHONE,1,1,Non-Target,0,DWELLING,Paid Search,MS,2025-01,Paid Search,Paid Search,NaN,Home - Conversion Tracking,1,0,0,0,0.0,0.0,0.0,0,0,Generic,google,Home
287359,2025-01-01,HOMEOWNERS,2025-01-01,2025-01,PHONE,2025,PHONE,1,1,Non-Target,0,DWELLING,Paid Search,NJ,2025-01,Paid Search,Paid Search,NaN,Home - Conversion Tracking,1,1,0,0,0.0,0.0,0.0,0,0,Generic,google,Home
287360,2025-01-01,HOMEOWNERS,2025-01-01,2025-01,PHONE,2025,PHONE,1,1,Non-Target,0,DWELLING,Paid Search,OK,2025-01,Paid Search,Paid Search,NaN,Home Call Extension - Conversion Tracking,1,1,0,0,0.0,0.0,0.0,0,0,Generic,google,Home
287361,2025-01-01,HOMEOWNERS,2025-01-01,2025-01,PHONE,2025,PHONE,1,1,Non-Target,0,DWELLING,Paid Search,TN,2025-01,Paid Search,Paid Search,NaN,Home Generic,1,1,0,0,0.0,0.0,0.0,0,0,Generic,google,Home
287362,2025-01-01,HOMEOWNERS,2025-01-01,2025-01,PHONE,2025,PHONE,1,1,Target,0,DWELLING,Paid Search,MA,2025-01,Paid Search,Paid Search,NaN,Home - Conversion Tracking,1,1,0,0,0.0,0.0,0.0,0,0,Generic,google,Home


In [374]:
internal_data.to_csv('/data/Users/Shelly/Search Report/Search_mtd.csv')

In [300]:
bing_data.head()

,lead_monyr,STATE,LINE,ENGINE,CAMPAIGN_NAME,KEYWORD_TYPE,IMPRESSIONS,CLICKS,SPEND,CONVERSIONS,Engine,keyword_type
0,2025-10,AZ,Auto,Bing,Auto Generic - AZ-Experiment - VBB,Generic,7637,247,20840.97,11,BING,Generic
1,2025-04,AZ,Enterprise,Bing,Amica Brand - Enterprise - Strategic - Bing,Brand,534,160,1257.87,41,BING,Brand
2,2025-11,TN,Enterprise,Bing,Amica Brand - Enterprise - Non Strategic - Bing,Brand,210,57,633.08,14,BING,Brand
3,2025-02,IN,Enterprise,Bing,Amica Brand - Enterprise - Non Strategic - Bin...,Brand,118,29,288.19,7,BING,Brand
4,2025-09,MA,Auto,Bing,Auto Generic - MA,Generic,24772,812,87784.10,196,BING,Generic


In [301]:
bing_data.CAMPAIGN_NAME.unique()

array(['Auto Generic - AZ-Experiment - VBB',
       'Amica Brand - Enterprise - Strategic - Bing',
       'Amica Brand - Enterprise - Non Strategic - Bing',
       'Amica Brand - Enterprise - Non Strategic - Bing-Experiment',
       'Auto Generic - MA', 'Home Generic - MD - Bing',
       'Amica Brand - Home Insurance - Bing', 'Amica Auto - Brand - Bing',
       'Home Generic - OR - Bing', 'Auto Generic - MD-Experiment - VBB',
       'Home Generic - NH - Bing', 'Auto Generic - NM',
       'Auto Generic - AZ-Experiment', 'Auto Generic - RI',
       'Auto Generic - PA', 'Auto Generic - MA - Broad Match Only',
       'Auto Generic - VA', 'Auto Generic - ME',
       'Auto Generic - PA - Bundle', 'Auto Generic - AZ',
       'Amica Brand - Enterprise - Strategic - Bing-Experiment',
       'Auto Generic - RI - PT Experiment', 'Home Generic - PA - Bing',
       'Auto Generic - CT - PT Experiment', 'Auto Generic - OR',
       'Home Generic - RI - Bing', 'Auto Generic - Non Strat 1',
       'Home

In [239]:
internal_data.head()

,business_date,line_of_business,lead_dt,bus_monyr,lead_contact_method,business_year,trans_contact_method,business_mon,business_day,state_grp,quality_seg,cov_item_type,lead_medium,state_cd,lead_monyr,last_touch_medium,medium_channel_adjusted,source,campaign,lead,quote,issue,prior_month_issue,day0,day7,same_month_issue
0,2024-01-01,AUTOMOBILE,2024-01-01,2024-01,WEB,2024,WEB,1,1,Tier2,5,PRIVATE PASSENGER,Affiliates and Aggregators,WA,2024-01,paid,Affiliates and Aggregators,ca,ca_auto_insurance,3,3,0,0,0.0,0.0,0.0
1,2024-01-01,AUTOMOBILE,2024-01-01,2024-01,WEB,2024,WEB,1,1,Tier2,5,PRIVATE PASSENGER,Paid Search,AZ,2024-01,cpc,Paid Search,google,auto generic - t3 - az,1,1,0,0,0.0,0.0,0.0
2,2024-01-01,AUTOMOBILE,2024-01-01,2024-01,WEB,2024,WEB,1,1,Tier2,5,PRIVATE PASSENGER,Paid Search,AZ,2024-01,unknown,Paid Search,NaN,Brand,1,1,0,0,0.0,0.0,0.0
3,2024-01-01,AUTOMOBILE,2024-01-01,2024-01,WEB,2024,WEB,1,1,Tier2,5,PRIVATE PASSENGER,Paid Search,DE,2024-01,cpc,Paid Search,google,auto generic - t3 - de,1,1,0,0,0.0,0.0,0.0
4,2024-01-01,AUTOMOBILE,2024-01-01,2024-01,WEB,2024,WEB,1,1,Tier2,5,PRIVATE PASSENGER,Paid Search,NC,2024-01,cpc,Paid Search,bing,amica brand - enterprise - t1 - pp,1,1,0,0,0.0,0.0,0.0


In [240]:
internal_data['campaign'].unique()

array(['ca_auto_insurance', 'auto generic - t3 - az', 'Brand',
       'auto generic - t3 - de', 'amica brand - enterprise - t1 - pp',
       'auto generic - t1 - nc', 'auto generic - t1 - ny',
       'auto generic - t1 - or', 'enterprise - t1 - pp - invest',
       'moneygroup_tier1pubs', 'auto generic - t1 - wa',
       'enterprise - t2 - non - strategic', nan, 'Generic',
       'auto generic - va - bing', 'home generic - t2 - va - bing',
       'auto generic - t2 - va',
       'Enterprise Call Extension - Conversion Tracking',
       'Google Auto Generic - Conversion Tracking',
       'Home Call Extension - Conversion Tracking', 'auto - brand',
       'Google Auto Call Extension - Conversion Tracking', 'PMAX',
       'auto generic - t1 - ma broad', 'auto generic - t1 - nh',
       'amica auto - brand - bing', 'auto generic - t1 - ct',
       'auto generic - t1 - ma', 'home generic - t1 - or',
       'home generic - t1 - wa', 'usnews_home_insurance',
       'home generic - t1 - ma', '

In [458]:
google_data.head()

,DATE,YEAR,MONTH,STATE,LINE,ENGINE,KEYWORD_TYPE,CAMPAIGN_NAME,CAMPAIGN_TYPE,IMPRESSIONS,SPEND,CLICKS,CONVERSIONS,Engine
0,2025-04-01,2025,4,NH,Home,Google Search,Generic,Home Generic - NH,SEARCH,307,790.56,10,0.0,Google
1,2025-08-12,2025,8,VA,Auto,Google Search,Generic,Auto Generic - VA,SEARCH,1392,5129.52,38,17.0,Google
2,2025-08-19,2025,8,OR,Auto,Google Search,Brand,Amica Auto - Brand,SEARCH,32,340.46,11,5.0,Google
3,2025-05-23,2025,5,NM,Auto,Google Search,Brand,Amica Auto - Brand,SEARCH,5,55.91,2,0.0,Google
4,2025-03-06,2025,3,AR,Enterprise,Google Search,Brand,Amica Brand - Enterprise - Non-Strategic,SEARCH,11,18.55,4,1.0,Google


,DATE,YEAR,MONTH,STATE,LINE,ENGINE,KEYWORD_TYPE,CAMPAIGN_NAME,CAMPAIGN_TYPE,IMPRESSIONS,SPEND,CLICKS,CONVERSIONS,Engine


In [609]:
internal_data[internal_data["CAMPAIGN"] == "Google Auto Generic - Conversion Tracking"].head()

,LEAD_STORAGE_NBR,LINE_OF_BUSINESS,QUALITY_SEG,LEAD_DT,LEAD_CONTACT_METHOD,LEAD_MEDIUM,MEDIUM_CHANNEL,MEDIUM,SOURCE,CAMPAIGN,LEAD,QUOTE,ISSUE,DAY0,DAY7,STATE_CD,STATE_GRP,LEAD_MONYR
262,Q007339684,HOMEOWNERS,0,2025-03-07,PHONE,Paid Search,Paid Search,Paid Search,None,Google Auto Generic - Conversion Tracking,1,1,0,0,0,NH,Target,2025-03
628,Q008983010,AUTOMOBILE,5,2026-01-08,PHONE,Paid Search,Paid Search,Paid Search,None,Google Auto Generic - Conversion Tracking,1,1,0,0,0,PA,Target,2026-01
753,Q007409723,HOMEOWNERS,0,2025-03-18,PHONE,Paid Search,Paid Search,Paid Search,None,Google Auto Generic - Conversion Tracking,1,1,0,0,0,NY,Target,2025-03
1625,Q007376155,AUTOMOBILE,4,2025-03-13,PHONE,Paid Search,Paid Search,Paid Search,None,Google Auto Generic - Conversion Tracking,1,1,0,0,0,NY,Target,2025-03
1701,Q007714878,AUTOMOBILE,5,2025-05-10,PHONE,Paid Search,Paid Search,Paid Search,None,Google Auto Generic - Conversion Tracking,1,1,0,0,0,WA,Tier2,2025-05
